# Qwen3.5 Visual-memory benchmark on RunPod

Runs**MST**, **color memory**, **associative inference** on Qwen3.5 to their dataset maximum, checks that the answers are real, and packages the results so they drop straight into her repo.

**"Note: Max context" here means the dataset maximum.** MST runs to 576 study images and color memory to 540, all the stimulus sets hold (the largest prompts are ~84k of the 262k-token context). On a single 80 GB card (an A100 or H100) the 27B stops at 250 images, because its two largest sizes need ~95–100 GB of GPU memory; section 15 explains how to finish them on a bigger GPU. Associative inference first goes through the pass/fail gate in `configs/assoc_inf_ab.yaml`; only a layout that passes gets a full sweep.

## 1. Settings

In [ ]:
import os
from pathlib import Path

# Where everything lives. /workspace is the pod's persistent volume.
WORK = Path(os.environ.get("BENCH_WORK", "/workspace"))
REPO = WORK / "qwen35-bench"
REPO_ZIP = WORK / "qwen35-bench-master.zip"       
GIT_URL = ""                                     
#GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "") 
HF_TOKEN = os.environ.get("HF_TOKEN", "") # optional: the Qwen3.5 checkpoints are public

# What to run
RUN_9B = True # MST and color memory on Qwen3.5-9B
RUN_27B = True # MST + color memory on Qwen3.5-27B
RUN_ASSOC_GATE = False # Associative-inference layout A/B on the 2B and 4B (configs/assoc_inf_ab.yaml)
GPUS = "0" # GPU(s) for the 9B, the gate and the smoke test
GPUS_27B = "0" # "0" on one GPU; "0,1" to shard the 27B over two 80 GB cards
PART_27B = "auto" # the 27B's MST 576 and color 540 need ~95-100 GB of GPU memory. "auto" runs them only if
# GPUS_27B has 110 GB+ (one H200, or two 80 GB cards), and otherwise stops the 27B at 250
# # images (section 15 finishes it later). Or choose yourself: "up_to_250" or "all".
FORCE_MAIN = False # True launches the full runs even if a preflight failed (not recommended)
PRICE_PER_HOUR = 0.0 # your pod's $/hour, for the cost estimate (optional)

# Derived paths (no need to change)
VENV = WORK / "envs" / "qwen35"
PY = VENV / "bin" / "python"
DATA = WORK / "memory_datasets"
HF_HOME = WORK / "hf_cache"
LOGS = WORK / "runpod_logs"
OUT = WORK / "outputs"

Helpers used by the cells below. Nothing to edit; run once per kernel session (and again after a kernel restart).

In [ ]:
import json, re, shutil, signal, subprocess, sys, time, urllib.request, zipfile
from datetime import datetime
from IPython.display import Image, Markdown, clear_output, display

ENV = dict(os.environ,
           HF_HOME=str(HF_HOME),
           PYTHONUNBUFFERED="1",
           TRITON_CACHE_DIR=str(WORK / ".triton"),       # compiled kernels survive restarts
           UV_CACHE_DIR=str(WORK / ".uv_cache"),
           UV_PYTHON_INSTALL_DIR=str(WORK / ".uv_python"))
if HF_TOKEN:
    ENV["HF_TOKEN"] = HF_TOKEN
RUN_ENV = {"HF_HUB_OFFLINE": "1"}   # benchmark jobs read weights from the local cache only, as on the cluster
for d in (LOGS, OUT, HF_HOME):
    d.mkdir(parents=True, exist_ok=True)

SIZES_GB = {"Qwen/Qwen3.5-0.8B": 2, "Qwen/Qwen3.5-2B": 5, "Qwen/Qwen3.5-4B": 9,
            "Qwen/Qwen3.5-9B": 19, "Qwen/Qwen3.5-27B": 55}


def models_needed():
    needed = ["Qwen/Qwen3.5-0.8B"]                       # smoke test
    if RUN_ASSOC_GATE:
        needed += ["Qwen/Qwen3.5-2B", "Qwen/Qwen3.5-4B"]
    if RUN_9B:
        needed.append("Qwen/Qwen3.5-9B")
    if RUN_27B:
        needed.append("Qwen/Qwen3.5-27B")
    return needed


# ---- what runs where ------------------------------------------------------------------------------
CONFIG_9B = "configs/runpod/qwen35_9b_mst_color.yaml"
CONFIG_27B_UPTO250 = "configs/runpod/qwen35_27b_mst_color_upto250.yaml"
CONFIG_27B_LARGEST = "configs/runpod/qwen35_27b_mst_color_largest.yaml"
PREFLIGHT_9B = ("configs/runpod/preflight_9b.yaml", {1, 540, 576})     # (config, its study sizes)
LARGEST_27B_GB = 110    # GPU memory for the 27B's MST 576 and color 540, which peak at ~95-100 GB


def gpu_memory():
    """{GPU index: GB}, in decimal GB like the result files' peak_gpu_gb (an "80 GB" A100 holds
    80 GiB = 85.9 GB). Empty without nvidia-smi."""
    smi = shutil.which("nvidia-smi")
    if not smi:
        return {}
    out = subprocess.run([smi, "--query-gpu=index,memory.total", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout
    mem = {}
    for line in out.strip().splitlines():
        try:
            index, mib = [x.strip() for x in line.split(",")]
            mem[int(index)] = float(mib) * 2**20 / 1e9
        except ValueError:
            pass
    return mem


def capacity(gpus):
    """GB of GPU memory across a GPU list such as "0" or "0,1"."""
    mem = gpu_memory()
    return sum(mem.get(int(i), 0) for i in str(gpus).split(","))


def plan_27b():
    """What this pod runs of the 27B: (full-run configs, (preflight config, its study sizes), largest sizes?)."""
    if PART_27B not in ("auto", "up_to_250", "all"):
        raise ValueError(f'PART_27B must be "auto", "up_to_250" or "all", not {PART_27B!r}')
    largest = PART_27B == "all" or (PART_27B == "auto" and capacity(GPUS_27B) >= LARGEST_27B_GB)
    if largest:
        return [CONFIG_27B_UPTO250, CONFIG_27B_LARGEST], ("configs/runpod/preflight_27b.yaml", {1, 540, 576}), True
    return [CONFIG_27B_UPTO250], ("configs/runpod/preflight_27b_upto250.yaml", {1, 250}), False


def run(cmd, cwd=None, check=True, echo=True):
    """Run a command (a list, or a shell string), streaming its output into the cell.
    Returns the exit code if it is non-zero, else None (so a cell ending in run(...) prints nothing extra)."""
    shell = isinstance(cmd, str)
    if not shell:
        cmd = [str(c) for c in cmd]
    if echo:
        print("$", cmd if shell else " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, shell=shell, cwd=cwd, env=ENV, text=True, bufsize=1,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError(f"command failed with exit code {rc}")
    return rc or None


def bench(*args, check=True):
    """Run the benchmark environment's Python from the repo root."""
    return run([PY, *args], cwd=REPO, check=check, echo=False)


def gpu_usage():
    smi = shutil.which("nvidia-smi")
    if not smi:
        return
    out = subprocess.run([smi, "--query-gpu=index,utilization.gpu,memory.used,memory.total",
                          "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout
    for line in out.strip().splitlines():
        i, util, used, total = [x.strip() for x in line.split(",")]
        print(f"GPU {i}: {util}% busy, {float(used) / 1024:.0f} of {float(total) / 1024:.0f} GiB in use")


# ---- background job queues --------------------------------------------------------------------
_QUEUES = {}   # name -> Popen. Polling it reaps a finished queue, which would otherwise linger as a zombie.


def _proc_fields(pid):
    """(state, ppid, pgid) of a live process, or None."""
    try:
        return Path(f"/proc/{pid}/stat").read_text().rsplit(")", 1)[1].split()[:3]
    except (OSError, IndexError):
        return None


def queue_running(name):
    p = _QUEUES.get(name)
    if p is not None:
        return p.poll() is None
    pid_file = LOGS / f"{name}.pid"  
    if not pid_file.exists():
        return False
    pid = int(pid_file.read_text())
    fields = _proc_fields(pid)
    if fields is None or fields[0] in ("Z", "X"):
        return False
    try:
        return b"run_queue.py" in Path(f"/proc/{pid}/cmdline").read_bytes()   # not a reused pid
    except OSError:
        return False


def launch(name, jobs):
    """Run jobs one after another in the background. Each job is (config, task or None, gpus)."""
    if queue_running(name):
        print(f"Queue '{name}' is already running; watch it with status('{name}').")
        return
    for config, _, _ in jobs:
        if not (REPO / config).exists():
            raise FileNotFoundError(f"{config} is not in {REPO}")
    queue_file = LOGS / f"{name}.queue.json"
    queue_file.write_text(json.dumps({
        "name": name, "python": str(PY), "repo": str(REPO), "log_dir": str(LOGS), "env": RUN_ENV,
        "jobs": [{"config": c, "task": t, "gpus": str(g)} for c, t, g in jobs]}, indent=2))
    (LOGS / f"{name}.status.json").unlink(missing_ok=True)
    log = open(LOGS / f"{name}.queue.log", "a")
    p = subprocess.Popen([str(PY), "runpod/run_queue.py", str(queue_file)], cwd=REPO, env=ENV,
                         stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    _QUEUES[name] = p
    (LOGS / f"{name}.pid").write_text(str(p.pid))
    print(f"Started queue '{name}' (pid {p.pid}). Jobs, in order:")
    for config, task, gpus in jobs:
        print(f"  {config}{' --task ' + task if task else ''}   GPU {gpus}")
    print(f"Logs: {LOGS}")


def status(name="main", tail=6):
    """One screen per queue: job states, cells done, the running job's last log lines, GPU use."""
    st_file = LOGS / f"{name}.status.json"
    running = queue_running(name)
    if not st_file.exists():
        print(f"Queue '{name}': " + ("starting..." if running else "never launched."))
        return
    st = json.loads(st_file.read_text())
    state = ("running" if running else "finished" if st.get("finished")
             else "stopped" if st.get("stopped") else "not running")
    print(f"Queue '{name}': {state}   started {st['started']}"
          + (f", finished {st['finished']}" if st.get("finished") else ""))
    specs = []
    for job in st["jobs"]:
        took = ""
        if job.get("start"):
            end = datetime.fromisoformat(job["end"]) if job.get("end") else datetime.now()
            took = str(end - datetime.fromisoformat(job["start"])).split(".")[0]
        task = f" --task {job['task']}" if job.get("task") else ""
        print(f"  [{job['state']:8s}] {job['config']}{task}   GPU {job['gpus']}   {took}")
        specs.append(job["config"] + (f":{job['task']}" if job.get("task") else ""))
    print()
    bench("runpod/progress.py", *specs, check=False)
    active = [j for j in st["jobs"] if j["state"] == "running"]
    if active and active[0].get("log") and Path(active[0]["log"]).exists():
        print(f"\nLast lines of {Path(active[0]['log']).name}:")
        for line in Path(active[0]["log"]).read_text(errors="replace").splitlines()[-tail:]:
            print("   ", line[:220])
        if not running:
            print("\nThe queue process is gone but a job is still marked running (pod restart, or stop()).\n"
                  "Launch the same queue again: finished cells are kept and skipped.")
    for j in st["jobs"]:
        if j["state"] == "failed":
            print(f"\nFAILED: {j['config']} (exit code {j['returncode']}). End of its log, {Path(j['log']).name}:")
            for line in Path(j["log"]).read_text(errors="replace").splitlines()[-8:]:
                print("   ", line[:220])
    gpu_usage()


def wait(name="main", every=60):
    """Refresh status(name) every `every` seconds until the queue finishes.
    Interrupting this cell (the stop button) only stops watching; the jobs keep running."""
    try:
        while queue_running(name):
            clear_output(wait=True)
            status(name)
            print(f"\nRefreshing every {every} s. Interrupt this cell any time; the jobs keep running.")
            time.sleep(every)
    except KeyboardInterrupt:
        print("Stopped watching. The jobs keep running; check them with status().")
        return
    clear_output(wait=True)
    status(name)


def _group_alive(pgid):
    for stat in Path("/proc").glob("[0-9]*/stat"):
        try:
            state, _, group = stat.read_text().rsplit(")", 1)[1].split()[:3]
        except (OSError, IndexError, ValueError):
            continue
        if group == str(pgid) and state not in ("Z", "X"):
            return True
    return False


def stop(name="main"):
    """Stop a queue and the job it is running. Finished cells are kept; launching again resumes."""
    if not queue_running(name):
        print(f"Queue '{name}' is not running.")
        return
    pgid = int((LOGS / f"{name}.pid").read_text())
    os.killpg(pgid, signal.SIGTERM)
    for _ in range(40):
        if not _group_alive(pgid):
            break
        time.sleep(0.5)
    else:
        try:
            os.killpg(pgid, signal.SIGKILL)
        except ProcessLookupError:
            pass
    if name in _QUEUES:
        _QUEUES.pop(name).poll()
    st_file = LOGS / f"{name}.status.json"
    if st_file.exists():
        st = json.loads(st_file.read_text())
        now = datetime.now().isoformat(timespec="seconds")
        for j in st["jobs"]:
            if j["state"] == "running":
                j.update(state="stopped", end=now)
        st["stopped"] = now
        st_file.write_text(json.dumps(st, indent=2))
    print(f"Stopped queue '{name}'.")


# ---- reading results ------------------------------------------------------------------------------
def show_replies(run_dir, k=3):
    """A few raw replies from each condition's largest cell, to eyeball for gibberish."""
    cells = {}
    for f in sorted((REPO / run_dir).glob("*/*.json")):
        d = json.loads(f.read_text())
        r, c = d["run"], d["run"]["condition"]
        key = (r["model"]["name"], r["task"]) + tuple(str(c.get(x, "")) for x in ("mode", "pair_type", "probe", "prompt"))
        if key not in cells or c.get("n_images", 0) > cells[key][0]:
            cells[key] = (c.get("n_images", 0), d)
    if not cells:
        print(f"No results under {run_dir}.")
        return
    for key, (n, d) in sorted(cells.items()):
        error = f"   ERROR: {d['error'].strip().splitlines()[-1][:150]}" if d.get("error") else ""
        print(f"\n{' '.join(x for x in key if x)}   n={n}{error}")
        for t in (d.get("trials") or [])[:k]:
            reply = t.get("free_response", t.get("raw_response"))
            mass = t.get("choice_mass")
            mass = "-" if mass is None else format(mass, ".3f")
            print(f"   target={str(t.get('target')):9s} answer={str(t.get('reported')):9s} "
                  f"mass={mass:6s} reply={repr(reply)[:80]}")


def result_problems(run_dir):
    """Cells whose result file records an error or a skip."""
    bad = []
    for f in sorted((REPO / run_dir).glob("*/*.json")):
        d = json.loads(f.read_text())
        if d.get("error"):
            bad.append(f"{f.parent.name}/{f.name}: {d['error'].strip().splitlines()[-1][:150]}")
        elif d.get("skipped"):
            bad.append(f"{f.parent.name}/{f.name}: skipped ({d['skipped']})")
    return bad


def preflight_verdict(model, gpus, sizes):
    """(passed, reasons) for one model's six preflight cells at the given study sizes."""
    cells = [json.loads(f.read_text()) for f in sorted((REPO / "results/runpod_preflight").glob("*/*.json"))]
    cells = [d for d in cells if d["run"]["model"]["name"] == model and d["run"]["condition"]["n_images"] in sizes]
    reasons, usable, replies = [], 0, 0
    if len(cells) < 6:
        reasons.append(f"only {len(cells)} of 6 preflight cells exist")
    cap = capacity(gpus)
    for d in cells:
        c = d["run"]["condition"]
        label = " ".join(x for x in (d["run"]["task"], c.get("mode", ""), f"n={c['n_images']}") if x)
        if d.get("error"):
            reasons.append(f"{label}: {d['error'].strip().splitlines()[-1][:150]}")
            continue
        if d.get("skipped"):
            reasons.append(f"{label}: skipped ({d['skipped']})")
            continue
        if cap and d.get("peak_gpu_gb", 0) > 0.95 * cap:
            reasons.append(f"{label}: peak {d['peak_gpu_gb']} GB is within 5% of the {cap:.0f} GB available")
        for t in d.get("trials", []):
            replies += 1
            if "raw_response" in t:               # continuous color: a number was parsed from the reply
                usable += t.get("reported", -1) != -1
            else:                                 # forced choice: the free-text reply contains a valid answer
                valid = [str(v) for v in t.get("answer_map", {"1": 0, "2": 0})]
                pattern = r"\b(" + "|".join(map(re.escape, valid)) + r")\b"
                usable += bool(re.search(pattern, t.get("free_response") or ""))
    if replies and usable < replies - 1:
        reasons.append(f"only {usable} of {replies} replies were usable answers (possible gibberish)")
    return not reasons, reasons


print("Helpers ready.")

Helpers ready.


In [3]:
RUN_ENV.update(OMP_NUM_THREADS="18", MKL_NUM_THREADS="18")

## 2. GPU, driver, disk and RAM

In [4]:
smi = shutil.which("nvidia-smi")
if smi is None:
    raise RuntimeError("nvidia-smi not found: this pod has no visible GPU.")
out = subprocess.run([smi, "--query-gpu=index,name,memory.total,driver_version", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True, check=True).stdout
GPU_MEM, driver = {}, "0"
for line in out.strip().splitlines():
    idx, name, mem, driver = [x.strip() for x in line.split(",")]
    GPU_MEM[int(idx)] = float(mem) * 2**20 / 1e9          # decimal GB, as in the result files
    print(f"GPU {idx}: {name}, {float(mem) / 1024:.0f} GiB = {GPU_MEM[int(idx)]:.1f} GB, driver {driver}")

fatal, warnings = [], []
if int(driver.split(".")[0]) < 570:
    fatal.append(f"Driver {driver} is older than 570; torch 2.10 with CUDA 12.8 needs 570+. "
                 "Deploy a new pod filtered to CUDA 12.8 or newer.")
for spec in {GPUS, GPUS_27B}:
    missing = [g for g in str(spec).split(",") if int(g) not in GPU_MEM]
    if missing:
        fatal.append(f"GPU {','.join(missing)} is in the settings but not on this pod.")
if RUN_9B and capacity(GPUS) < 45:
    warnings.append(f"The 9B needs ~35-40 GB at its largest prompts; GPUS={GPUS!r} gives {capacity(GPUS):.0f} GB.")
if RUN_27B:
    cap = capacity(GPUS_27B)
    largest = plan_27b()[2]
    if cap < 80:
        warnings.append(f"The 27B needs ~75 GB even at 250 images; GPUS_27B={GPUS_27B!r} gives {cap:.0f} GB. "
                        "The preflight will show whether it fits.")
    elif largest and cap < LARGEST_27B_GB:
        warnings.append(f'PART_27B is "all", but MST 576 and color 540 need ~95-100 GB and GPUS_27B={GPUS_27B!r} '
                        f"gives {cap:.0f} GB. The preflight will show whether they fit.")
    if largest:
        print(f"\n27B: every size, up to MST 576 and color 540 ({cap:.0f} GB on GPUS_27B={GPUS_27B!r}).")
    elif PART_27B == "up_to_250":
        print('\n27B: sizes 1 to 250 only (PART_27B is "up_to_250"). MST 576 and color 540 are left for later '
              "(section 15 explains how to finish them).")
    else:
        print(f"\n27B: sizes 1 to 250 only. MST 576 and color 540 need ~95-100 GB and GPUS_27B={GPUS_27B!r} has "
              f"{cap:.0f} GB, so they are left for a bigger GPU (section 15 explains how to finish them).")

pending = [m for m in models_needed() if not (HF_HOME / "hub" / ("models--" + m.replace("/", "--"))).exists()]
need = sum(SIZES_GB[m] for m in pending) + (0 if PY.exists() else 12) + 5
free = shutil.disk_usage(WORK).free / 1e9
print(f"\nDisk: {free:.0f} GB free in {WORK}; about {need:.0f} GB still needed (weights, environment, headroom)")
if free < need:
    fatal.append(f"Not enough disk in {WORK}: {free:.0f} GB free, about {need:.0f} GB needed. Use a bigger volume "
                 "(150 GB covers everything), or set RUN_27B = False for this pod.")

ram = int(re.search(r"MemTotal:\s+(\d+)", Path("/proc/meminfo").read_text()).group(1)) / 1e6
for limit_file in ("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory/memory.limit_in_bytes"):
    try:
        value = Path(limit_file).read_text().strip()
        if value.isdigit():
            ram = min(ram, int(value) / 1e9)
    except OSError:
        pass
print(f"RAM: {ram:.0f} GB")
if ram < 16:
    warnings.append(f"Only {ram:.0f} GB of RAM; the largest episodes decode ~1,700 images at once. 16 GB+ is safer.")

print()
for w in warnings:
    print("WARNING:", w)
if fatal:
    raise RuntimeError("Fix before continuing:\n  - " + "\n  - ".join(fatal))
print("Checks passed." if not warnings else "No blocking problems.")

GPU 0: NVIDIA H200, 140 GiB = 150.8 GB, driver 595.91.07

27B: every size, up to MST 576 and color 540 (151 GB on GPUS_27B='0').

Disk: 368856 GB free in /workspace; about 93 GB still needed (weights, environment, headroom)
RAM: 251 GB

Checks passed.


## 3. Repo

In [5]:
if (REPO / "run_bench.py").exists():
    print(f"Repo already at {REPO}")
elif GIT_URL:
    url = GIT_URL
    if GITHUB_TOKEN and url.startswith("https://"):
        url = url.replace("https://", f"https://x-access-token:{GITHUB_TOKEN}@", 1)
    print(f"Cloning {GIT_URL} ...")
    run(["git", "clone", "--depth", "1", url, REPO], echo=False)     # not echoed: the URL may hold a token
elif REPO_ZIP.exists():
    with zipfile.ZipFile(REPO_ZIP) as z:
        top = z.namelist()[0].split("/")[0]
        z.extractall(WORK)
        commit = z.comment.decode(errors="replace").strip()          # GitHub archives carry the commit here
    (WORK / top).rename(REPO)
    (REPO / "RUNPOD_SOURCE.json").write_text(json.dumps(
        {"source": REPO_ZIP.name, "archive_commit": commit or None,
         "unpacked": datetime.now().isoformat(timespec="seconds")}, indent=2))
else:
    raise FileNotFoundError(f"No repo yet. Upload qwen35-bench-master.zip to {WORK} "
                            "(drag it into the Jupyter file browser), or set GIT_URL in the settings cell.")

for p in ("run_bench.py", "bench/runner.py", "configs/assoc_inf_ab.yaml", "configs/ported_tasks_test_0.8b.yaml",
          "requirements.txt", "requirements.lock.txt"):
    assert (REPO / p).exists(), f"{p} is missing from the repo"
if (REPO / ".git").exists():
    commit = subprocess.run(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"],
                            capture_output=True, text=True).stdout.strip()
    print("Repo at commit", commit)
elif (REPO / "RUNPOD_SOURCE.json").exists():
    print("Unpacked from", json.loads((REPO / "RUNPOD_SOURCE.json").read_text()))
pickups = sorted((REPO / "pickups").glob("PICKUP_*.md"))
if pickups:
    print("Latest pickup note in the repo:", pickups[-1].name)

Repo already at /workspace/qwen35-bench
Unpacked from {'source': 'qwen35-bench-master.zip', 'archive_commit': '6c1090c08b3dd8a3975d18d56bf268f5e3f2e7cb', 'unpacked': '2026-10-11T03:30:45'}
Latest pickup note in the repo: PICKUP_2026_10_06.md


If you uploaded a results zip from an earlier run of this notebook (`qwen35_runpod_results_*.zip`) to `/workspace`, this copies its finished result files into the repo, so those cells are skipped below. Files already in the repo are never overwritten. With no such zip it does nothing.

In [6]:
previous = sorted(WORK.glob("qwen35_runpod_results_*.zip"))
for zip_path in previous:
    restored = kept = 0
    with zipfile.ZipFile(zip_path) as z:
        for name in z.namelist():
            if not (name.startswith("results/") and name.endswith(".json")):
                continue
            if (REPO / name).exists():
                kept += 1
            else:
                z.extract(name, REPO)
                restored += 1
    print(f"{zip_path.name}: restored {restored} result files" + (f", kept {kept} already here" if kept else ""))
if not previous:
    print(f"No earlier results zip in {WORK}; nothing to restore.")

No earlier results zip in /workspace; nothing to restore.


## 4. Python environment

`setup_env.sh` and `requirements.lock.txt`, in its own Python 3.12 environment on the volume (the kernel's Python is left alone). The first build takes about 5–10 minutes; afterwards this cell only checks it.

In [ ]:
CAUSAL_CONV1D_WHEEL = ("https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.7.0/"
                       "causal_conv1d-1.7.0+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl")
CHECK = ("import torch, transformers, fla, causal_conv1d; assert torch.cuda.is_available(), 'CUDA not available'; "
         "print('torch', torch.__version__, '| transformers', transformers.__version__, "
         "'| GPU', torch.cuda.get_device_name(0))")


def env_ok():
    if not PY.exists():
        return False
    r = subprocess.run([str(PY), "-c", CHECK], env=ENV, capture_output=True, text=True)
    print((r.stdout + r.stderr).strip()[-1500:])
    return r.returncode == 0


if env_ok():
    print("Environment ready:", VENV)
else:
    run([sys.executable, "-m", "pip", "install", "-q", "uv"])
    uv_bin = Path(sys.executable).parent / "uv"
    uv = [str(uv_bin)] if uv_bin.exists() else [sys.executable, "-m", "uv"]
    run([*uv, "venv", "--python", "3.12", VENV])
    run([*uv, "pip", "install", "--python", PY, "torch==2.10.0", "torchvision==0.25.0",
         "--index-url", "https://download.pytorch.org/whl/cu128"])
    run([*uv, "pip", "install", "--python", PY, CAUSAL_CONV1D_WHEEL])
    # Everything else pinned to the lock file (torch, CUDA and causal-conv1d came from the steps above).
    lock = (REPO / "requirements.lock.txt").read_text().splitlines()
    pins = [l for l in lock if l.strip() and not re.match(r"(torch|torchvision|triton|nvidia-|cuda-|causal-conv1d)\b", l)]
    (WORK / "constraints.txt").write_text("\n".join(pins) + "\n")
    run([*uv, "pip", "install", "--python", PY, "-r", REPO / "requirements.txt", "-c", WORK / "constraints.txt"])
    assert env_ok(), "The environment check failed; read the output above."
    print("Environment ready:", VENV)

$ /usr/local/bin/python -m pip install -q uv
$ /usr/local/bin/uv venv --python 3.12 /workspace/envs/qwen35
Using CPython 3.12.3 interpreter at: /usr/local/bin/python
Creating virtual environment at: envs/qwen35
Activate with: source envs/qwen35/bin/activate
$ /usr/local/bin/uv pip install --python /workspace/envs/qwen35/bin/python torch==2.10.0 torchvision==0.25.0 --index-url https://download.pytorch.org/whl/cu128
Using Python 3.12.3 environment at: envs/qwen35
Resolved 31 packages in 396ms
Prepared 31 packages in 33.66s
Installed 31 packages in 2.80s
 + cuda-bindings==12.9.4
 + cuda-pathfinder==1.6.0
 + filelock==3.32.3
 + fsspec==2026.7.0
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.6.1
 + numpy==2.5.2
 + nvidia-cublas-cu12==12.8.4.1
 + nvidia-cuda-cupti-cu12==12.8.90
 + nvidia-cuda-nvrtc-cu12==12.8.93
 + nvidia-cuda-runtime-cu12==12.8.90
 + nvidia-cudnn-cu12==9.10.2.21
 + nvidia-cufft-cu12==11.3.3.83
 + nvidia-cufile-cu12==1.13.1.3
 + nvidia-curand-cu12==10.

## 5. Stimuli

Brady 2008 objects (2,400 plus exemplar and state pairs), Brady 2013 color objects (540), MST sets 1–6, and the 1,637-word pool. THINGS is not needed for these tasks.

In [ ]:
STIMULI = {   # file: (url, size in bytes). raw.githubusercontent.com is where GitHub's "raw" links redirect.
    "memory_datasets_brady.zip": ("https://raw.githubusercontent.com/csiyer/vision-memory/main/memory_datasets_brady.zip", 80955797),
    "memory_datasets_mst.zip": ("https://raw.githubusercontent.com/csiyer/vision-memory/main/memory_datasets_mst.zip", 63189250),
}
EXPECTED = {"Brady2008Objects": 2400, "Brady2008Exemplar": 200, "Brady2008State": 200,
            "Brady2013ColorObjects": 540, **{f"MST/Set {i}": 384 for i in range(1, 7)}}


def download(url, target, size, attempts=4):
    """Download to a .part file, retrying cut-off transfers; a complete file of the wrong size is an error."""
    part = target.with_name(target.name + ".part")
    for attempt in range(1, attempts + 1):
        try:
            urllib.request.urlretrieve(url, part)
        except OSError as e:                    # includes truncated transfers (ContentTooShortError)
            print(f"  attempt {attempt} of {attempts} failed ({e.__class__.__name__}); retrying", flush=True)
            time.sleep(5 * attempt)
            continue
        if part.stat().st_size != size:
            raise RuntimeError(f"{target.name} is {part.stat().st_size} bytes, expected {size}: "
                               "the source repo may have changed, so check before running.")
        part.replace(target)
        return
    raise RuntimeError(f"Could not download {url} after {attempts} attempts.")


def count_images(folder):
    if not folder.is_dir():
        return 0
    return sum(1 for p in folder.iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png"))


def stimuli_report():
    counts = {k: count_images(DATA / k) for k in EXPECTED}
    pool = DATA / "wasnorm_wordpool.txt"
    words = len(pool.read_text().split()) if pool.exists() else 0
    bins = len(list((DATA / "MST").glob("Set* bins.txt"))) if (DATA / "MST").exists() else 0
    ok = all(counts[k] == v for k, v in EXPECTED.items()) and words >= 1637 and bins == 6
    return ok, counts, words, bins


ok, *_ = stimuli_report()
if not ok:
    downloads = WORK / "downloads"
    downloads.mkdir(exist_ok=True)
    for name, (url, size) in STIMULI.items():
        target = downloads / name
        if not (target.exists() and target.stat().st_size == size):
            print(f"Downloading {name} ({size / 1e6:.0f} MB) ...", flush=True)
            download(url, target, size)
        with zipfile.ZipFile(target) as z:
            z.extractall(WORK)               # both zips hold a top-level memory_datasets/ folder

ok, counts, words, bins = stimuli_report()
for k, v in counts.items():
    print(f"  {k:24s} {v:5d} images (expected {EXPECTED[k]})")
print(f"  word pool: {words} words; MST similarity-bin files: {bins} of 6")
assert ok, "The stimuli are incomplete; see the counts above."

link = REPO / "memory_datasets"              # the tasks read memory_datasets/ relative to the repo root
if link.is_symlink() and link.resolve() != DATA.resolve():
    link.unlink()
if not link.exists() and not link.is_symlink():
    link.symlink_to(DATA)
print(f"\n{link} -> {link.resolve()}")

  Brady2008Objects          2400 images (expected 2400)
  Brady2008Exemplar          200 images (expected 200)
  Brady2008State             200 images (expected 200)
  Brady2013ColorObjects      540 images (expected 540)
  MST/Set 1                  384 images (expected 384)
  MST/Set 2                  384 images (expected 384)
  MST/Set 3                  384 images (expected 384)
  MST/Set 4                  384 images (expected 384)
  MST/Set 5                  384 images (expected 384)
  MST/Set 6                  384 images (expected 384)
  word pool: 1638 words; MST similarity-bin files: 6 of 6

/workspace/qwen35-bench/memory_datasets -> /workspace/memory_datasets


## 6. Model weights

About 2 GB (0.8B), 5 GB (2B), 9 GB (4B), 19 GB (9B) and 55 GB (27B), cached in `/workspace/hf_cache`. The benchmark jobs then run offline, as on the cluster. Already-downloaded files are skipped.

In [9]:
for repo_id in models_needed():
    print(f"\n== {repo_id}", flush=True)
    bench("-c", f"from huggingface_hub import snapshot_download; print(snapshot_download({repo_id!r}))")
print("\nWeights ready.")


== Qwen/Qwen3.5-0.8B

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Fetching 13 files: 100%|██████████| 13/13 [00:01<00:00,  6.91it/s]
/workspace/hf_cache/hub/models--Qwen--Qwen3.5-0.8B/snapshots/2fc06364715b967f1860aea9cf38778875588b17

== Qwen/Qwen3.5-9B

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Fetching 16 files: 100%|██████████| 16/16 [00:06<00:00,  2.37it/s]
/workspace/hf_cache/hub/models--Qwen--Qwen3.5-9B/snapshots/c202236235762e1c871ad0ccb60c8ee5ba337b9a

== Qwen/Qwen3.5-27B

Fetching 24 files:   0%|          | 0/24 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Fetching 2

## 7. Configs and helper scripts

Six configs and seven small scripts, written into the repo under `configs/runpod/` and `runpod/`. 

The full-run configs use exactly the sizes, answer layout and trial counts of her 4B config (2 seeds × 50 trials per cell), and keep her run names, so the results pool into her existing `results/qwen35_9b` and `results/qwen35_27b`. The 27B's sizes are split over two configs, 1 to 250 (fits one 80 GB card) and MST 576 plus color 540 (needs ~95–100 GB), which together are the 4B's sizes. The preflight configs run one trial per cell at n=1 and at the largest n the pod will run, under a separate run name so they never pool with the real cells.

In [10]:
(REPO / "configs" / "runpod").mkdir(parents=True, exist_ok=True)
(REPO / "runpod").mkdir(parents=True, exist_ok=True)
print("ok")

ok


In [11]:
%%writefile {REPO}/configs/runpod/qwen35_9b_mst_color.yaml
# Qwen3.5-9B: MST and color memory, the two tasks configs/qwen35_9b.yaml left out ("add them once
# these three are read"). Same sizes, layout and trial counts as configs/qwen35_4b.yaml, so the line
# 0.8B -> 2B -> 4B -> 9B is like for like. Written for RunPod.
#
# Both tasks are dataset-bound, not context-bound: MST holds 1,152 pairs (half are needed as foils)
# and Brady2013ColorObjects 540 images. At max_image_size 512 an MST or color image costs ~146
# tokens, so the largest episodes are ~84k (MST 576) and ~79k (color 540) tokens, a third of the
# 262k context. The 4B peaked at 24.2 / 25.5 GB on them; the 9B adds ~9.5 GB of weights, so expect
# ~35-40 GB. Any 80 GB card is comfortable. Preflight first (configs/runpod/preflight_9b.yaml).
#
# Results pool into results/qwen35_9b/ beside the first three tasks (same run_name).

run_name: qwen35_9b

output_dir: results

seeds: [0, 1]

models:
  - name: qwen3.5-9b
    model_id: Qwen/Qwen3.5-9B
    max_image_size: 512
    max_new_tokens: 100

tasks:
  - task: mst
    n_images: [1, 5, 25, 100, 250, 576]     # 1,152 MST pairs, half needed as foils
    prompts: [numbered]
    n_trials: 50

  - task: color_memory
    modes: [named, continuous]
    n_images: [1, 5, 25, 100, 250, 540]     # Brady2013ColorObjects holds 540 images
    prompts: [numbered]
    n_trials: 50


Overwriting /workspace/qwen35-bench/configs/runpod/qwen35_9b_mst_color.yaml


In [12]:
%%writefile {REPO}/configs/runpod/qwen35_27b_mst_color_upto250.yaml
# Qwen3.5-27B: MST and color memory up to 250 study images, the fifth point on the scaling line.
# Same layout and trial counts as configs/qwen35_4b.yaml and the 9B companion. Written for RunPod.
#
# The 27B's sizes are split over two files so that one 80 GB card can run most of them:
#   this file                            n = 1, 5, 25, 100, 250   peaks at ~75 GB (n=250 is ~38k tokens)
#   qwen35_27b_mst_color_largest.yaml    MST 576, color 540       peaks at ~95-100 GB (~84k / ~79k tokens)
# Together they are exactly the 4B's sizes. Weights are 54.71 GB and each 1k tokens adds ~0.5 GB
# (88.3 GB at 65k tokens on 2 x A6000, LIMITATIONS.md). Both files keep run_name qwen35_27b, so their
# results pool with each other and with the first three tasks.
#
# 27B sharded over 8 x A6000 once produced gibberish on every trial (job 13066794, cause unknown).
# On any new hardware read raw replies from the preflight before the full run.

run_name: qwen35_27b

output_dir: results

seeds: [0, 1]

models:
  - name: qwen3.5-27b
    model_id: Qwen/Qwen3.5-27B
    max_image_size: 512
    max_new_tokens: 100
    device_map: auto

tasks:
  - task: mst
    n_images: [1, 5, 25, 100, 250]
    prompts: [numbered]
    n_trials: 50

  - task: color_memory
    modes: [named, continuous]
    n_images: [1, 5, 25, 100, 250]
    prompts: [numbered]
    n_trials: 50


Overwriting /workspace/qwen35-bench/configs/runpod/qwen35_27b_mst_color_upto250.yaml


In [13]:
%%writefile {REPO}/configs/runpod/qwen35_27b_mst_color_largest.yaml
# Qwen3.5-27B: MST and color memory at the dataset maxima, MST 576 and color 540. Companion to
# qwen35_27b_mst_color_upto250.yaml, which has the memory numbers; together they are the 4B's sizes.
#
# These prompts are ~84k tokens (MST 576) and ~79k (color 540) and peak at ~95-100 GB: one 141 GB
# H200 fits them without sharding, a single 80 GB A100 or H100 does not. On two 80 GB cards
# `device_map: auto` shards the model, but accelerate fills card 0 first, so watch the first card,
# not the total.
#
# 27B sharded over 8 x A6000 once produced gibberish on every trial (job 13066794, cause unknown).
# On any new hardware read raw replies from the preflight before the full run.
#
# Results pool into results/qwen35_27b/ beside the smaller sizes (same run_name).

run_name: qwen35_27b

output_dir: results

seeds: [0, 1]

models:
  - name: qwen3.5-27b
    model_id: Qwen/Qwen3.5-27B
    max_image_size: 512
    max_new_tokens: 100
    device_map: auto

tasks:
  - task: mst
    n_images: [576]     # 1,152 MST pairs, half needed as foils
    prompts: [numbered]
    n_trials: 50

  - task: color_memory
    modes: [named, continuous]
    n_images: [540]     # Brady2013ColorObjects holds 540 images
    prompts: [numbered]
    n_trials: 50


Overwriting /workspace/qwen35-bench/configs/runpod/qwen35_27b_mst_color_largest.yaml


In [14]:
%%writefile {REPO}/configs/runpod/preflight_9b.yaml
# One trial per cell at n=1 and at each task's largest n, before the full 9B run. Reads peak GPU
# memory, seconds per trial and the raw replies on this pod's hardware. The n=1 cells come first,
# so the one-off Triton compile lands there and the largest-n timings stay clean.
# Separate run_name: these 1-trial files must not pool with the final 50-trial cells.

run_name: runpod_preflight

output_dir: results

seeds: [0]

models:
  - name: qwen3.5-9b
    model_id: Qwen/Qwen3.5-9B
    max_image_size: 512
    max_new_tokens: 100

tasks:
  - task: mst
    n_images: [1, 576]
    prompts: [numbered]
    n_trials: 1

  - task: color_memory
    modes: [named, continuous]
    n_images: [1, 540]
    prompts: [numbered]
    n_trials: 1


Overwriting /workspace/qwen35-bench/configs/runpod/preflight_9b.yaml


In [15]:
%%writefile {REPO}/configs/runpod/preflight_27b.yaml
# One trial per cell at n=1 and at each task's largest n, before a 27B run that includes the largest
# sizes (a pod with 110 GB+ of GPU memory: one H200, or two 80 GB cards; on one 80 GB card the
# notebook uses preflight_27b_upto250.yaml instead). Reads peak GPU memory, seconds per trial and
# the raw replies on this pod's hardware (the 27B has produced gibberish on one cluster setup, so
# the replies matter as much as the memory).
# Separate run_name: these 1-trial files must not pool with the final 50-trial cells.

run_name: runpod_preflight

output_dir: results

seeds: [0]

models:
  - name: qwen3.5-27b
    model_id: Qwen/Qwen3.5-27B
    max_image_size: 512
    max_new_tokens: 100
    device_map: auto

tasks:
  - task: mst
    n_images: [1, 576]
    prompts: [numbered]
    n_trials: 1

  - task: color_memory
    modes: [named, continuous]
    n_images: [1, 540]
    prompts: [numbered]
    n_trials: 1


Overwriting /workspace/qwen35-bench/configs/runpod/preflight_27b.yaml


In [16]:
%%writefile {REPO}/configs/runpod/preflight_27b_upto250.yaml
# One trial per cell at n=1 and n=250, before the 27B run on a card too small for its largest sizes
# (one 80 GB A100 or H100). 250 is the largest size such a pod runs (configs/runpod/
# qwen35_27b_mst_color_upto250.yaml, ~75 GB); MST 576 and color 540 are left for a bigger GPU.
# Reads peak GPU memory, seconds per trial and the raw replies on this pod's hardware (the 27B has
# produced gibberish on one cluster setup, so the replies matter as much as the memory).
# Separate run_name: these 1-trial files must not pool with the final 50-trial cells.

run_name: runpod_preflight

output_dir: results

seeds: [0]

models:
  - name: qwen3.5-27b
    model_id: Qwen/Qwen3.5-27B
    max_image_size: 512
    max_new_tokens: 100
    device_map: auto

tasks:
  - task: mst
    n_images: [1, 250]
    prompts: [numbered]
    n_trials: 1

  - task: color_memory
    modes: [named, continuous]
    n_images: [1, 250]
    prompts: [numbered]
    n_trials: 1


Overwriting /workspace/qwen35-bench/configs/runpod/preflight_27b_upto250.yaml


| Script | What it does |
|---|---|
| `run_queue.py` | runs jobs one after another in the background, with a status file and one log per job |
| `progress.py` | done / total result files per config; also validates every size against the datasets, without loading a model |
| `preflight_report.py` | peak GPU memory, timing and sample replies from the preflight, plus a time and cost estimate |
| `check_results.py` | the gibberish checks: reply format, agreement with the forced choice, parse failures, comparison with the 4B |
| `assoc_gate.py` | applies the associative-inference gate exactly as written in `configs/assoc_inf_ab.yaml` |
| `plot_extra.py` | figures for MST, color memory and associative inference (`bench.plot` covers only the first three tasks) |
| `rescore_color.py` | optional: re-scores the old continuous-color results with the corrected parser (open item 59) |

In [17]:
%%writefile {REPO}/runpod/run_queue.py
#!/usr/bin/env python
"""Run benchmark jobs one after another, detached from the notebook.

Usage: python runpod/run_queue.py <queue.json>

queue.json holds {"name", "python", "repo", "log_dir", "env", "jobs": [{"config", "task", "gpus"}]}.
Each job is `python run_bench.py <config> [--task <task>]` with CUDA_VISIBLE_DEVICES set to its GPUs;
a job without a task runs every task in its config, loading the model once.
The status file <log_dir>/<name>.status.json is rewritten after every change, and each job gets
its own log. A failed job does not stop the queue. The runner skips result files that already
exist, so starting the same queue again resumes where it stopped.
"""
import json
import os
import subprocess
import sys
from datetime import datetime
from pathlib import Path


def now():
    return datetime.now().isoformat(timespec="seconds")


def main():
    queue = json.loads(Path(sys.argv[1]).read_text())
    log_dir = Path(queue["log_dir"])
    log_dir.mkdir(parents=True, exist_ok=True)
    status_path = log_dir / f"{queue['name']}.status.json"
    status = {"name": queue["name"], "pid": os.getpid(), "started": now(), "finished": None,
              "jobs": [{**job, "state": "pending", "start": None, "end": None,
                        "returncode": None, "log": None} for job in queue["jobs"]]}

    def save():
        tmp = status_path.with_suffix(".tmp")
        tmp.write_text(json.dumps(status, indent=2))
        tmp.replace(status_path)

    save()
    for i, job in enumerate(status["jobs"]):
        log = log_dir / f"{queue['name']}_{i:02d}_{Path(job['config']).stem}_{job.get('task') or 'all'}.log"
        job.update(state="running", start=now(), log=str(log))
        save()
        env = {**os.environ, **queue.get("env", {}), "CUDA_VISIBLE_DEVICES": str(job.get("gpus", "0"))}
        cmd = [queue["python"], "run_bench.py", job["config"]]
        if job.get("task"):
            cmd += ["--task", job["task"]]
        cmd += job.get("extra_args", [])
        with open(log, "a") as f:
            f.write(f"# {now()} {' '.join(cmd)}  (CUDA_VISIBLE_DEVICES={env['CUDA_VISIBLE_DEVICES']})\n")
            f.flush()
            returncode = subprocess.call(cmd, cwd=queue["repo"], env=env, stdout=f, stderr=subprocess.STDOUT)
        job.update(state="done" if returncode == 0 else "failed", end=now(), returncode=returncode)
        save()
    status["finished"] = now()
    save()


if __name__ == "__main__":
    main()


Overwriting /workspace/qwen35-bench/runpod/run_queue.py


In [18]:
%%writefile {REPO}/runpod/progress.py
#!/usr/bin/env python
"""Done / total result files for each config, using the runner's own run expansion.

Usage (from the repo root): python runpod/progress.py [--json] <config.yaml>[:<task>] [...]

bench.runner imports torch, so its done-test is repeated here to keep this check fast and usable
while a job holds the GPU. A result counts as done exactly as in bench.runner.is_done: the file
exists, parses, has no error, and was not skipped. Running this also validates each config's sizes
against the datasets (expand_runs raises on a size the data cannot supply), without loading a model.
--json prints one JSON list ({config, task, done, failed, skipped, todo, total} per spec) instead.
"""
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from bench.config import expand_runs, load_config  # noqa: E402


def state(path):
    if not path.exists():
        return "todo"
    try:
        result = json.loads(path.read_text())
    except json.JSONDecodeError:
        return "todo"
    if result.get("error") is not None:
        return "failed"
    if result.get("skipped"):
        return "skipped"
    return "done"


def main():
    args = sys.argv[1:]
    as_json = "--json" in args
    rows = []
    for spec in (a for a in args if a != "--json"):
        config_path, _, task = spec.partition(":")
        cfg = load_config(config_path)
        if task:
            cfg["tasks"] = [t for t in cfg["tasks"] if t["task"] == task]
        runs = expand_runs(cfg)
        states = [state(Path(cfg["output_dir"]) / cfg["run_name"] / r.task / r.result_name) for r in runs]
        counts = {s: states.count(s) for s in ("done", "failed", "skipped", "todo")}
        rows.append({"config": config_path, "task": task or None, **counts, "total": len(runs)})
        if not as_json:
            label = config_path + (f" --task {task}" if task else "")
            print(f"{label:58s} {counts['done']:4d}/{len(runs):<4d} done   "
                  f"failed={counts['failed']} skipped={counts['skipped']} todo={counts['todo']}")
    if as_json:
        print(json.dumps(rows))


if __name__ == "__main__":
    main()


Overwriting /workspace/qwen35-bench/runpod/progress.py


In [19]:
%%writefile {REPO}/runpod/preflight_report.py
#!/usr/bin/env python
"""Read the one-trial preflight cells and estimate how long the full runs will take.

Usage (from the repo root):
    python runpod/preflight_report.py results/runpod_preflight \
        --configs configs/runpod/qwen35_9b_mst_color.yaml configs/runpod/qwen35_27b_mst_color.yaml \
        [--capacity '{"qwen3.5-9b": 80, "qwen3.5-27b": 141}'] [--price 3.5]

Timing model (rough, for planning only). A preflight cell makes one capacity-probe call plus one
trial; a trial is two calls for forced-choice tasks (score_choices, then free text) and one for
continuous color. Treating every call as costing the same, call(n) = elapsed / (1 + calls per trial),
interpolated linearly in n between the n=1 and largest-n cells. The first cell of a job also pays
the one-off Triton compile, so each model's small-prompt cost is the minimum n=1 call over its tasks.
"""
import argparse
import json
import sys
from collections import defaultdict
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from bench.config import expand_runs, load_config  # noqa: E402


def calls_per_trial(task, condition):
    return 1 if (task == "color_memory" and condition.get("mode") == "continuous") else 2


def kind(task, condition):
    return f"{task}/{condition['mode']}" if task == "color_memory" else task


def sample_reply(result):
    trials = result.get("trials") or []
    if not trials:
        return ""
    t = trials[0]
    text = t.get("free_response", t.get("raw_response"))
    return repr((text or "")[:60])


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("preflight_dir")
    parser.add_argument("--configs", nargs="*", default=[])
    parser.add_argument("--capacity", default="{}", help="JSON {model name: GB available to that model}")
    parser.add_argument("--price", type=float, default=0.0, help="pod price in $/hour")
    args = parser.parse_args()
    capacity = json.loads(args.capacity)

    cells = [json.loads(p.read_text()) for p in sorted(Path(args.preflight_dir).glob("*/*.json"))]
    if not cells:
        print(f"No preflight results under {args.preflight_dir}.")
        return

    print(f"{'model':13s} {'task':24s} {'n':>5s} {'status':8s} {'peak GB':>8s} {'cap GB':>7s} {'s':>7s} "
          f"{'mass':>6s}  reply")
    call_time = defaultdict(dict)       # (model, kind) -> {n: seconds per call}
    problems = []
    for d in sorted(cells, key=lambda d: (d["run"]["model"]["name"], d["run"]["task"],
                                          d["run"]["condition"].get("mode", ""), d["run"]["condition"]["n_images"])):
        model, task, cond = d["run"]["model"]["name"], d["run"]["task"], d["run"]["condition"]
        k, n = kind(task, cond), cond["n_images"]
        cap = capacity.get(model)
        if d.get("error"):
            status = "FAILED"
            last = d["error"].strip().splitlines()[-1]
            problems.append(f"{model} {k} n={n}: {last}")
        elif d.get("skipped"):
            status = "SKIPPED"
            problems.append(f"{model} {k} n={n}: {d['skipped']}")
        else:
            status = "ok"
            call_time[(model, k)][n] = d["elapsed_s"] / (1 + calls_per_trial(task, cond))
            if cap and d["peak_gpu_gb"] > 0.9 * cap:
                problems.append(f"{model} {k} n={n}: peak {d['peak_gpu_gb']} GB is within 10% of {cap} GB")
        masses = [t["choice_mass"] for t in d.get("trials", []) if "choice_mass" in t]
        mass = f"{min(masses):.3f}" if masses else "-"
        print(f"{model:13s} {k:24s} {n:5d} {status:8s} {d.get('peak_gpu_gb', 0):8.1f} {cap or '-':>7} "
              f"{d.get('elapsed_s', 0):7.1f} {mass:>6s}  {sample_reply(d)}")

    if problems:
        print("\nNeeds attention before the full run:")
        for p in problems:
            print("  -", p)
    else:
        print("\nNo failures, skips or near-full GPUs.")

    if not args.configs:
        return
    floor = defaultdict(lambda: None)
    for (model, k), times in call_time.items():
        if 1 in times:
            floor[model] = times[1] if floor[model] is None else min(floor[model], times[1])

    print("\nEstimated wall time for the full configs (rough):")
    total_hours = 0.0
    for config_path in args.configs:
        cfg = load_config(config_path)
        per_model = defaultdict(float)
        missing = set()
        for run in expand_runs(cfg):
            model, cond = run.model["name"], run.condition
            k, n = kind(run.task, cond), cond["n_images"]
            times = call_time.get((model, k), {})
            big = max(times) if times else None
            if big is None or floor[model] is None:
                missing.add(f"{model} {k}")
                continue
            c1, cmax = floor[model], times[big]
            call = c1 + (cmax - c1) * max(0, n - 1) / max(1, big - 1)
            per_model[model] += (1 + cond["n_trials"] * calls_per_trial(run.task, cond)) * call
        for model, seconds in per_model.items():
            hours = seconds / 3600
            total_hours += hours
            print(f"  {config_path}: {model} ~{hours:.1f} h")
        for m in sorted(missing):
            print(f"  {config_path}: no preflight timing for {m}, not estimated")
    if total_hours:
        cost = f", about ${total_hours * args.price:.0f} at ${args.price}/h" if args.price else ""
        print(f"  total ~{total_hours:.1f} h{cost} (run sequentially on one pod)")


if __name__ == "__main__":
    main()


Overwriting /workspace/qwen35-bench/runpod/preflight_report.py


In [20]:
%%writefile {REPO}/runpod/check_results.py
#!/usr/bin/env python
"""Check that results are real answers and not gibberish, then write a short report.

Usage (from the repo root):
    python runpod/check_results.py results/qwen35_9b results/qwen35_27b [--report out.md] [--csv out.csv]

Per cell (one model x task x condition, pooled over seeds) it reports bench.summarize's numbers and
flags (accuracy, Wilson 95% CI, chance, LOW_MASS, BIAS) and adds four checks of its own:
  - wellformed: share of free-text replies that are exactly one valid answer (e.g. "2").
  - agree: share of trials whose free-text answer matches the forced choice read off the logits.
    The forced choice always yields an answer; agreement shows it is the model's real answer.
  - continuous color: parse-failure rate and mean absolute error in degrees.
  - BELOW_4B?: a 9B/27B cell more than 10 points under the 4B at the same n (LIMITATIONS.md
    reference values), which more likely means a broken run than a worse model.
Flags added here: GIBBERISH? (wellformed and agree both < 0.5), DISAGREE (agree < 0.8),
PARSE (continuous parse failures > 20%), BELOW_4B?.
"""
import argparse
import csv
import json
import re
import sys
from collections import defaultdict
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from bench.summarize import load_rows  # noqa: E402

# Qwen3.5-4B, numbered layout, 100 episodes per cell (LIMITATIONS.md, "MST, color memory and harder foils").
REF_4B_ACC = {
    "mst": {1: 0.75, 5: 0.76, 25: 0.62, 100: 0.58, 250: 0.43, 576: 0.40},
    "color_memory/named": {1: 0.86, 540: 0.38},
}
REF_4B_MAE = {1: 45.0}      # continuous color; from n=5 up the 4B is at chance (82-91 degrees)
KEY_FIELDS = ("model", "task", "dataset", "foil_type", "prompt", "mode", "pair_type", "probe", "n_images")
EXTRA_DEFAULTS = {"peak_gb": None, "gpus": [], "s_per_trial": None, "wellformed": None, "agree": None,
                  "mae": None, "parse_fail": None, "ldi": None}


def cell_key(model, task, cond):
    values = {"model": model, "task": task, **cond}
    return tuple(values.get(f, "") if f != "n_images" else values.get(f) for f in KEY_FIELDS)


def kind(task, mode):
    return f"{task}/{mode}" if task == "color_memory" else task


def first_answer(text, valid):
    """The first token in the reply that is a valid answer, or None."""
    if text is None:
        return None
    for token in re.findall(r"[A-Za-z]+|\d+", text):
        if token.lower() in valid:
            return token.lower()
    return None


def extras_by_cell(run_dirs):
    groups = defaultdict(list)
    for path in sorted(p for d in run_dirs for p in Path(d).glob("*/*.json")):
        d = json.loads(path.read_text())
        cond = {k: v for k, v in d["run"]["condition"].items() if k != "n_trials"}
        groups[cell_key(d["run"]["model"]["name"], d["run"]["task"], cond)].append(d)

    out = {}
    for key, results in groups.items():
        trials = [t for r in results for t in r.get("trials", [])]
        e = dict(EXTRA_DEFAULTS)
        e["peak_gb"] = max((r.get("peak_gpu_gb") or 0) for r in results)
        e["gpus"] = sorted({(r.get("provenance") or {}).get("gpu") or "?" for r in results})
        e["s_per_trial"] = (sum(r.get("elapsed_s") or 0 for r in results) / len(trials)) if trials else None
        free = [t for t in trials if "free_response" in t]
        if free:
            wellformed = agree = 0
            for t in free:
                if "answer_map" in t:              # mst, named color: the forced choice is t["choice"]
                    valid, forced = {str(k).lower() for k in t["answer_map"]}, str(t["choice"]).lower()
                else:                              # associative inference: options 1 and 2
                    valid, forced = {"1", "2"}, str(t["reported"])
                reply = (t["free_response"] or "").strip().strip(".").lower()
                wellformed += reply in valid
                agree += first_answer(t["free_response"], valid) == forced
            e["wellformed"], e["agree"] = wellformed / len(free), agree / len(free)
        if trials and "abs_error_degrees" in trials[0]:
            errors = [t["abs_error_degrees"] for t in trials if t["abs_error_degrees"] is not None]
            e["mae"] = sum(errors) / len(errors) if errors else None
            e["parse_fail"] = sum(t["reported"] == -1 for t in trials) / len(trials)
        ldis = [(r["metrics"]["ldi"], len(r["trials"])) for r in results
                if r.get("metrics") and "ldi" in r["metrics"] and r.get("trials")]
        if ldis:
            e["ldi"] = sum(v * w for v, w in ldis) / sum(w for _, w in ldis)
        out[key] = e
    return out


def add_flags(row):
    flags = [f for f in row["flags"].split(",") if f]
    if row["errors"]:
        flags.append("ERROR")
    if row["skipped"]:
        flags.append("SKIPPED")
    if row["wellformed"] is not None:
        if row["wellformed"] < 0.5 and row["agree"] < 0.5:
            flags.append("GIBBERISH?")
        elif row["agree"] < 0.8:
            flags.append("DISAGREE")
    if row["parse_fail"] is not None and row["parse_fail"] > 0.2:
        flags.append("PARSE")
    big = any(s in row["model"] for s in ("9b", "27b"))
    k, n = kind(row["task"], row["mode"]), row["n_images"]
    ref = REF_4B_ACC.get(k, {}).get(n)
    if big and ref is not None and row["accuracy"] < ref - 0.10:
        flags.append("BELOW_4B?")
    if big and k == "color_memory/continuous" and n in REF_4B_MAE and row["mae"] is not None \
            and row["mae"] > REF_4B_MAE[n] + 15:
        flags.append("BELOW_4B?")
    return flags


def fmt(v, spec=".2f"):
    return "-" if v is None else format(v, spec)


def table(rows):
    """Plain-text table grouped by task, the same text going to stdout and the report."""
    lines = []
    by_kind = defaultdict(list)
    for r in rows:
        by_kind[kind(r["task"], r["mode"])].append(r)
    for k in sorted(by_kind):
        lines.append(f"\n### {k}")
        if k == "color_memory/continuous":
            lines.append(f"{'model':13s} {'n':>5s} {'MAE deg':>8s} {'4B ref':>7s} {'parse fail':>10s} "
                         f"{'within30':>8s}  flags")
            for r in by_kind[k]:
                ref = REF_4B_MAE.get(r["n_images"])
                lines.append(f"{r['model']:13s} {r['n_images']:5d} {fmt(r['mae'], '.1f'):>8s} "
                             f"{fmt(ref, '.0f'):>7s} {fmt(r['parse_fail']):>10s} {r['accuracy']:8.2f}  "
                             f"{','.join(r['all_flags'])}")
            continue
        assoc = k == "associative_inference"
        head = f"{'model':13s} "
        head += f"{'layout':9s} {'pair':5s} {'probe':10s} {'chains':>6s} " if assoc else f"{'n':>5s} "
        head += f"{'acc':>5s} {'95% CI':>12s} {'chance':>6s} {'4B ref':>6s} "
        head += f"{'LDI':>6s} " if k == "mst" else ""
        head += f"{'wellf':>5s} {'agree':>5s}  flags"
        lines.append(head)
        for r in by_kind[k]:
            line = f"{r['model']:13s} "
            if assoc:
                chains = r["n_images"] // {"image": 4, "word": 3}.get(r["pair_type"], 1)
                line += f"{r['prompt']:9s} {r['pair_type']:5s} {r['probe']:10s} {chains:6d} "
            else:
                line += f"{r['n_images']:5d} "
            ref = REF_4B_ACC.get(k, {}).get(r["n_images"])
            line += f"{r['accuracy']:5.2f} [{r['ci95_low']:.2f},{r['ci95_high']:.2f}] {r['chance']:6.2f} {fmt(ref):>6s} "
            line += f"{fmt(r['ldi']):>6s} " if k == "mst" else ""
            line += f"{fmt(r['wellformed']):>5s} {fmt(r['agree']):>5s}  {','.join(r['all_flags'])}"
            lines.append(line)
    return lines


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("run_dirs", nargs="+")
    parser.add_argument("--report", help="write a markdown report here")
    parser.add_argument("--csv", help="write every cell with its checks here")
    args = parser.parse_args()

    dirs = [d for d in args.run_dirs if Path(d).exists()]
    rows = load_rows(*dirs) if dirs else []
    if not rows:
        print("No results found in", args.run_dirs)
        return
    extras = extras_by_cell(dirs)
    for r in rows:
        r.update(EXTRA_DEFAULTS)
        r.update(extras.get(tuple(r.get(f, "") if f != "n_images" else r.get(f) for f in KEY_FIELDS), {}))
        r["all_flags"] = add_flags(r)

    lines = table(rows)
    counts = defaultdict(int)
    for r in rows:
        for f in r["all_flags"]:
            counts[f] += 1
    summary = [f"\n{len(rows)} cells checked. Flag counts: " +
               (", ".join(f"{f} {c}" for f, c in sorted(counts.items())) or "none")]
    serious = sum(any(f in r["all_flags"] for f in ("ERROR", "GIBBERISH?", "PARSE", "LOW_MASS", "BELOW_4B?"))
                  for r in rows)
    summary.append("Cells needing a look (ERROR, GIBBERISH?, PARSE, LOW_MASS, BELOW_4B?): "
                   f"{serious}. BIAS on its own is expected where a model is at chance.")
    print("\n".join(lines + summary))

    if args.csv:
        with open(args.csv, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()), extrasaction="ignore")
            writer.writeheader()
            for r in rows:
                writer.writerow({**r, "all_flags": ",".join(r["all_flags"]), "gpus": ";".join(r["gpus"])})
        print(f"wrote {args.csv}")

    if args.report:
        prov = defaultdict(int)
        for path in sorted(p for d in dirs for p in Path(d).glob("*/*.json")):
            p = json.loads(path.read_text()).get("provenance") or {}
            prov[(p.get("gpu"), p.get("torch"), p.get("transformers"))] += 1
        md = ["# Qwen3.5 benchmark: RunPod results check", "",
              f"Run directories: {', '.join(dirs)}", "",
              "Environment (GPU, torch, transformers: result files):"]
        md += [f"- {g}, torch {t}, transformers {tr}: {n}" for (g, t, tr), n in sorted(prov.items(), key=str)]
        md += ["", "Columns: acc with Wilson 95% CI; chance; the 4B reference from LIMITATIONS.md where one "
               "exists; wellf = free-text replies that are exactly one valid answer; agree = free-text answer "
               "matches the forced choice. Flags as in the script header.", "", "```"]
        md += [line for line in lines if line] + ["```"] + summary
        Path(args.report).write_text("\n".join(md) + "\n")
        print(f"wrote {args.report}")


if __name__ == "__main__":
    main()


Overwriting /workspace/qwen35-bench/runpod/check_results.py


In [21]:
%%writefile {REPO}/runpod/assoc_gate.py
#!/usr/bin/env python
"""Apply the associative-inference gate written in configs/assoc_inf_ab.yaml before it was run.

Usage (from the repo root):
    python runpod/assoc_gate.py results/assoc_inf_ab [--json out.json] [--write-config configs/runpod/assoc_inf_full.yaml]

Gate (copied from the config header): a (layout, pair_type) is usable only if, for BOTH the 2B and
4B, at n_chains 2 and 5, direct_ab and direct_bc are above chance (95% Wilson lower bound > 0.5) and
neither is flagged BIAS (one answer on > 80% of trials). Only then is its inference curve read as
memory and a full sweep proposed. The prediction recorded with it: if image->image binding is the
failure, direct_ab fails for both pair types while direct_bc passes for `word`.
"""
import argparse
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from bench.summarize import load_rows  # noqa: E402

IMAGES_PER_CHAIN = {"image": 4, "word": 3}
GATE_MODELS = ("qwen3.5-2b", "qwen3.5-4b")
GATE_CHAINS = (2, 5)
GATE_PROBES = ("direct_ab", "direct_bc")

# Full sweep for a passing layout: up to the dataset limit on Brady2008 (800 image chains, 1,200 word
# chains). The largest cells are 210k-250k tokens, so the 9B wants a 141 GB H200 there (its near-full-
# context peak is ~78 GB), and the 27B (about 190 GB near full context) needs two; it is left out.
FULL_LADDER = {"image": [2, 5, 25, 100, 250, 500, 800], "word": [2, 5, 25, 100, 250, 500, 800, 1200]}
FULL_MODELS = [("qwen3.5-0.8b", "Qwen/Qwen3.5-0.8B"), ("qwen3.5-2b", "Qwen/Qwen3.5-2B"),
               ("qwen3.5-4b", "Qwen/Qwen3.5-4B"), ("qwen3.5-9b", "Qwen/Qwen3.5-9B")]


def chains(row):
    return row["n_images"] // IMAGES_PER_CHAIN[row["pair_type"]]


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("run_dir")
    parser.add_argument("--json", help="write the gate result here")
    parser.add_argument("--write-config", help="if any layout passes, write a full-sweep config here")
    args = parser.parse_args()

    rows = [r for r in load_rows(args.run_dir) if r["task"] == "associative_inference"] \
        if Path(args.run_dir).exists() else []
    if not rows:
        print(f"No associative-inference results in {args.run_dir}.")
        return
    index = {(r["model"], r["prompt"], r["pair_type"], r["probe"], chains(r)): r for r in rows}
    layouts = sorted({r["prompt"] for r in rows})
    pair_types = sorted({r["pair_type"] for r in rows})

    def ok(model, layout, pair_type, probe, n):
        r = index.get((model, layout, pair_type, probe, n))
        return r is not None and r["above_chance"] and "BIAS" not in r["flags"]

    results = []
    print("Gate: direct_ab and direct_bc above chance (Wilson lower bound > 0.5) with no BIAS flag,")
    print("for both the 2B and 4B, at 2 and 5 chains.\n")
    for layout in layouts:
        for pair_type in pair_types:
            failures = []
            for model in GATE_MODELS:
                for n in GATE_CHAINS:
                    for probe in GATE_PROBES:
                        r = index.get((model, layout, pair_type, probe, n))
                        if r is None:
                            failures.append(f"{model} {probe} {n}ch: missing")
                        elif not ok(model, layout, pair_type, probe, n):
                            why = "BIAS" if "BIAS" in r["flags"] else "at chance"
                            failures.append(f"{model} {probe} {n}ch: {r['accuracy']:.2f} "
                                            f"(low {r['ci95_low']:.2f}, top answer {r['most_common_frac']}) {why}")
            passed = not failures
            results.append({"layout": layout, "pair_type": pair_type, "passed": passed, "failures": failures})
            print(f"{layout:9s} {pair_type:5s} {'PASS' if passed else 'FAIL'}")
            for f in failures:
                print(f"      {f}")

    print("\nInference probe (read as memory only where the layout passed):")
    for layout in layouts:
        for pair_type in pair_types:
            cells = []
            for model in GATE_MODELS:
                for n in sorted({chains(r) for r in rows if r["pair_type"] == pair_type}):
                    r = index.get((model, layout, pair_type, "inference", n))
                    if r:
                        bias = " BIAS" if "BIAS" in r["flags"] else ""
                        cells.append(f"{model.split('-')[-1]}@{n}: {r['accuracy']:.2f}{bias}")
            print(f"  {layout:9s} {pair_type:5s} " + "  ".join(cells))

    def direct(pair_type, probe):
        """Did this probe pass for both models at both gate sizes, under any layout?"""
        return any(all(ok(m, layout, pair_type, probe, n) for m in GATE_MODELS for n in GATE_CHAINS)
                   for layout in layouts)

    if {"image", "word"} <= set(pair_types):
        predicted = (not direct("image", "direct_ab") and not direct("word", "direct_ab")
                     and direct("word", "direct_bc"))
        print("\nRecorded prediction (image->image binding is the failure: direct_ab fails for both pair "
              f"types, direct_bc passes for word): {'matches' if predicted else 'does not match'} the data.")

    passing = [r for r in results if r["passed"]]
    if args.json:
        Path(args.json).write_text(json.dumps({"results": results, "passing": passing}, indent=2))
        print(f"wrote {args.json}")
    if passing and args.write_config:
        lines = ["# Full associative-inference sweep for the layout(s) that passed the gate",
                 f"# ({', '.join(p['layout'] + '/' + p['pair_type'] for p in passing)}), written by runpod/assoc_gate.py.",
                 "# The largest cells are 210k-250k tokens: the 9B wants a 141 GB H200 there. The 27B is left out",
                 "# (about 190 GB near full context); add it only on two H200s. Preflight before the full run.",
                 "", "run_name: assoc_inf_full", "", "output_dir: results", "", "seeds: [0, 1]", "", "models:"]
        for name, model_id in FULL_MODELS:
            lines += [f"  - name: {name}", f"    model_id: {model_id}", "    max_image_size: 512",
                      "    max_new_tokens: 100"]
        lines += ["", "tasks:"]
        for p in passing:
            lines += ["  - task: associative_inference", "    datasets: [Brady2008]",
                      f"    pair_types: [{p['pair_type']}]", "    probe_types: [inference, direct_ab, direct_bc]",
                      f"    n_chains: {FULL_LADDER[p['pair_type']]}", f"    prompts: [{p['layout']}]",
                      "    n_trials: 50", ""]
        Path(args.write_config).parent.mkdir(parents=True, exist_ok=True)
        Path(args.write_config).write_text("\n".join(lines))
        print(f"wrote {args.write_config}")
    elif not passing:
        print("\nNo layout passed, so there is no full sweep to run. Record this in the pickup note:")
        print("associative inference still has no working test format after the re-port.")


if __name__ == "__main__":
    main()


Overwriting /workspace/qwen35-bench/runpod/assoc_gate.py


In [22]:
%%writefile {REPO}/runpod/plot_extra.py
#!/usr/bin/env python
"""Figures for the three tasks bench.plot does not draw: MST, color memory, associative inference.

Usage (from the repo root): python runpod/plot_extra.py <run_dir> [<run_dir> ...] [--out plots/runpod]

Same conventions as bench/plot.py: log-x study length, Wilson 95% CIs, one hue light -> dark by
model size, markers as a second cue. Adjacent blues in that ramp are hard to tell apart (the 4B and
9B pair fails a normal-vision separation check), so each line also carries a direct end label in
neutral ink. The 4B's published values (LIMITATIONS.md) are drawn as grey crosses for reference,
since the small-model result files live on the cluster, not on this pod.
"""
import argparse
import json
import sys
from collections import defaultdict
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
from matplotlib.ticker import FuncFormatter, NullLocator  # noqa: E402

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from bench.summarize import load_rows  # noqa: E402

MODELS = {"qwen3.5-0.8b": ("#86b6ef", "o"), "qwen3.5-2b": ("#5598e7", "s"), "qwen3.5-4b": ("#2a78d6", "v"),
          "qwen3.5-9b": ("#1c5cab", "^"), "qwen3.5-27b": ("#0d366b", "D")}
INK, MUTED, GRID = "#222222", "#777777", "#dddddd"
REF_4B_ACC = {"mst": {1: 0.75, 5: 0.76, 25: 0.62, 100: 0.58, 250: 0.43, 576: 0.40},
              "named": {1: 0.86, 540: 0.38}}
REF_4B_LDI = {1: 0.35, 5: 0.29, 25: 0.00, 100: 0.21, 250: 0.14, 576: 0.23}
REF_4B_MAE = {1: 45.0}
PROBE_STYLE = {"inference": "-", "direct_ab": "--", "direct_bc": ":"}


def style(model):
    return MODELS.get(model, ("#888888", "x"))


def short(model):
    return model.split("-")[-1].upper()


def model_order(models):
    order = list(MODELS)
    return sorted(models, key=lambda m: order.index(m) if m in order else len(order))


def int_ticks(ax, values):
    """Ticks at the sizes actually run, labelled as plain numbers (log axes default to 10^k)."""
    values = sorted(set(values))
    ax.set_xticks(values)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{int(round(v)):,}"))
    ax.xaxis.set_minor_locator(NullLocator())


def end_labels(ax, ends):
    """Direct labels at the line ends, nudged apart so two lines ending together stay readable.
    Call after the axis limits are final; `ends` is [(x, y, model)]."""
    if not ends:
        return
    lo, hi = ax.get_ylim()
    gap = 0.055 * (hi - lo)
    placed = []
    for x, y, model in sorted(ends, key=lambda e: e[1]):
        if placed and y - placed[-1][1] < gap:
            y = placed[-1][1] + gap
        placed.append((x, y, model))
    overflow = placed[-1][1] - hi
    if overflow > 0:
        placed = [(x, y - overflow, m) for x, y, m in placed]
    for x, y, model in placed:
        ax.annotate(short(model), (x, y), xytext=(7, 0), textcoords="offset points", va="center",
                    fontsize=9, color=INK, annotation_clip=False)


def frame(ax, xs, xlabel="Study sequence length"):
    ax.set_xscale("log")
    int_ticks(ax, xs)
    lo, hi = min(xs), max(xs)
    ax.set_xlim(lo / 1.4, hi * 1.9)           # room on the right for the end labels
    ax.set_xlabel(xlabel)
    ax.grid(color=GRID, lw=0.8)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)


def acc_axes(ax, rows, chance, ref=None, label_chance="Chance"):
    by_model = defaultdict(list)
    for r in rows:
        by_model[r["model"]].append(r)
    xs = [r["n_images"] for r in rows] + (list(ref) if ref else [])
    ends = []
    for model in model_order(by_model):
        rs = sorted(by_model[model], key=lambda r: r["n_images"])
        color, marker = style(model)
        x = [r["n_images"] for r in rs]
        y = [100 * r["accuracy"] for r in rs]
        err = [[100 * (r["accuracy"] - r["ci95_low"]) for r in rs], [100 * (r["ci95_high"] - r["accuracy"]) for r in rs]]
        ax.errorbar(x, y, yerr=err, color=color, marker=marker, ms=7, capsize=3, label=model, lw=2)
        ends.append((x[-1], y[-1], model))
    if ref:
        ax.plot(list(ref), [100 * v for v in ref.values()], "x", color=MUTED, ms=8, mew=2,
                label="Qwen3.5-4B (published)")
    ax.axhline(100 * chance, ls="--", color="#999999", lw=1, label=label_chance)
    frame(ax, xs)
    ax.set_ylim(0, 105)
    ax.set_ylabel("Accuracy (%)")
    end_labels(ax, ends)


def line_axes(ax, values, ylabel, chance=None, ref=None, chance_label="Chance", ylim=None):
    by_model = defaultdict(dict)
    for (model, n), v in values.items():
        by_model[model][n] = v
    ends = []
    for model in model_order(by_model):
        color, marker = style(model)
        ns = sorted(by_model[model])
        ys = [by_model[model][n] for n in ns]
        ax.plot(ns, ys, color=color, marker=marker, ms=7, lw=2, label=model)
        ends.append((ns[-1], ys[-1], model))
    if ref:
        ax.plot(list(ref), list(ref.values()), "x", color=MUTED, ms=8, mew=2, label="Qwen3.5-4B (published)")
    if chance is not None:
        ax.axhline(chance, ls="--", color="#999999", lw=1, label=chance_label)
    frame(ax, [n for (_, n) in values] + (list(ref) if ref else []))
    if ylim:
        ax.set_ylim(*ylim)
    ax.set_ylabel(ylabel)
    end_labels(ax, ends)


def save(fig, ax, path):
    handles, labels = ax.get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=3, frameon=True, bbox_to_anchor=(0.5, -0.02))
    fig.tight_layout(rect=(0, 0.12, 1, 1))
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("wrote", path)


def per_cell_metric(run_dirs, task, mode, metric):
    """Trial-weighted mean of a per-file metric (ldi) or of a per-trial field (mae), per (model, n)."""
    sums = defaultdict(lambda: [0.0, 0])
    for path in sorted(p for d in run_dirs for p in Path(d).glob(f"{task}/*.json")):
        d = json.loads(path.read_text())
        cond = d["run"]["condition"]
        if mode and cond.get("mode") != mode:
            continue
        key = (d["run"]["model"]["name"], cond["n_images"])
        if metric == "ldi" and d.get("metrics") and "ldi" in d["metrics"] and d.get("trials"):
            sums[key][0] += d["metrics"]["ldi"] * len(d["trials"])
            sums[key][1] += len(d["trials"])
        elif metric == "mae":
            errors = [t["abs_error_degrees"] for t in d.get("trials", []) if t.get("abs_error_degrees") is not None]
            sums[key][0] += sum(errors)
            sums[key][1] += len(errors)
    return {k: s / n for k, (s, n) in sums.items() if n}


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("run_dirs", nargs="+")
    parser.add_argument("--out", default="plots/runpod")
    args = parser.parse_args()
    dirs = [d for d in args.run_dirs if Path(d).exists()]
    if not dirs:
        print("No run directories found in", args.run_dirs)
        return
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    rows = load_rows(*dirs)

    mst = [r for r in rows if r["task"] == "mst"]
    if mst:
        fig, ax = plt.subplots(figsize=(7, 5))
        acc_axes(ax, mst, 1 / 3, REF_4B_ACC["mst"])
        ax.set_title("MST (old / similar / new), numbered layout\nAccuracy vs sequence length")
        save(fig, ax, out / "mst_accuracy.png")
        ldi = per_cell_metric(dirs, "mst", None, "ldi")
        if ldi:
            fig, ax = plt.subplots(figsize=(7, 5))
            line_axes(ax, ldi, "LDI = p(similar | lure) - p(similar | foil)", 0.0, REF_4B_LDI, "No discrimination")
            ax.set_title("MST lure discrimination index")
            save(fig, ax, out / "mst_ldi.png")

    named = [r for r in rows if r["task"] == "color_memory" and r["mode"] == "named"]
    if named:
        fig, ax = plt.subplots(figsize=(7, 5))
        acc_axes(ax, named, 1 / 6, REF_4B_ACC["named"])
        ax.set_title("Color memory, named (6 colors), numbered layout\nAccuracy vs sequence length")
        save(fig, ax, out / "color_named_accuracy.png")

    mae = per_cell_metric(dirs, "color_memory", "continuous", "mae")
    if mae:
        fig, ax = plt.subplots(figsize=(7, 5))
        line_axes(ax, mae, "Mean absolute error (degrees)", 90.0, REF_4B_MAE, "Chance (90 degrees)", ylim=(0, 180))
        ax.set_title("Color memory, continuous report\nError vs sequence length (lower is better)")
        save(fig, ax, out / "color_continuous_error.png")

    assoc = [r for r in rows if r["task"] == "associative_inference"]
    if assoc:
        layouts = sorted({r["prompt"] for r in assoc})
        pair_types = sorted({r["pair_type"] for r in assoc})
        per_chain = {"image": 4, "word": 3}
        fig, axes = plt.subplots(len(pair_types), len(layouts), figsize=(4.5 * len(layouts), 3.8 * len(pair_types)),
                                 squeeze=False, sharey=True)
        for i, pair_type in enumerate(pair_types):
            for j, layout in enumerate(layouts):
                ax = axes[i][j]
                cells = defaultdict(list)
                for r in assoc:
                    if r["pair_type"] == pair_type and r["prompt"] == layout:
                        cells[(r["model"], r["probe"])].append(r)
                xs = []
                for (model, probe), rs in sorted(cells.items()):
                    rs.sort(key=lambda r: r["n_images"])
                    color, marker = style(model)
                    x = [r["n_images"] // per_chain[pair_type] for r in rs]
                    xs += x
                    ax.plot(x, [100 * r["accuracy"] for r in rs], color=color, marker=marker, ms=6,
                            ls=PROBE_STYLE.get(probe, "-"), lw=1.8, label=f"{model} {probe}")
                ax.axhline(50, ls="--", color="#999999", lw=1)
                frame(ax, xs or [1], xlabel="Chains studied")
                ax.set_ylim(0, 105)
                ax.set_title(f"{layout} / {pair_type} pairs", fontsize=10)
            axes[i][0].set_ylabel("Accuracy (%)")
        save(fig, axes[0][0], out / "assoc_inference.png")


if __name__ == "__main__":
    main()


Overwriting /workspace/qwen35-bench/runpod/plot_extra.py


In [23]:
%%writefile {REPO}/runpod/rescore_color.py
#!/usr/bin/env python
"""Re-score continuous color-memory results with the corrected parser (open item 59). CPU only.

Usage (from the repo root): python runpod/rescore_color.py <in_dir> <out_dir>

The 0.8B/2B/4B continuous cells were scored with a parser that took the first 1-3 digit run in the
reply. bench/tasks/color_memory_eval.parse_degrees now takes the last number in [0, 360], as
vision-memory does. This re-parses each stored raw_response with it, recomputes the trial errors and
the cell metrics exactly as the task does, and writes copies under out_dir (inputs are not touched).
Each trial keeps its old value as `reported_old_parser`. Expected changes per LIMITATIONS.md:
0.8B 105, 2B 83, 4B 0 of 1,600 trials.
"""
import json
import sys
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from bench.tasks.color_memory_eval import parse_degrees  # noqa: E402
from src.metrics import calculate_color_metrics  # noqa: E402


def main():
    in_dir, out_dir = Path(sys.argv[1]), Path(sys.argv[2])
    changed_by_model, total_by_model = defaultdict(int), defaultdict(int)
    for path in sorted(in_dir.rglob("*.json")):
        d = json.loads(path.read_text())
        run = d.get("run", {})
        if run.get("task") != "color_memory" or run.get("condition", {}).get("mode") != "continuous":
            continue
        trials = d.get("trials", [])
        changed = 0
        for t in trials:
            new = parse_degrees(t.get("raw_response"))
            if new != t["reported"]:
                changed += 1
            t["reported_old_parser"] = t["reported"]
            t["reported"] = new
            error = None if new == -1 else abs((new - t["target"] + 180) % 360 - 180)
            t["abs_error_degrees"] = error
            t["correct"] = None if error is None else int(error <= 30)
        scored = [t for t in trials if t["reported"] != -1]
        metrics = (calculate_color_metrics([t["reported"] for t in scored], [t["target"] for t in scored])
                   if scored else {"accuracy": 0})
        if trials:
            metrics["parse_failure_rate"] = sum(t["reported"] == -1 for t in trials) / len(trials)
            metrics["frac_within_30_degrees"] = sum(bool(t["correct"]) for t in trials) / len(trials)
            metrics["response_distribution"] = dict(Counter(str(t["reported"]) for t in trials).most_common())
        d["metrics"] = metrics
        d["rescored"] = {"parser": "parse_degrees: last number in [0, 360]", "changed_trials": changed,
                         "at": datetime.now().isoformat(timespec="seconds")}
        target = out_dir / path.relative_to(in_dir)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(json.dumps(d, indent=2))
        model = run["model"]["name"]
        changed_by_model[model] += changed
        total_by_model[model] += len(trials)
    if not total_by_model:
        print(f"No continuous color-memory results under {in_dir}.")
    for model in sorted(total_by_model):
        print(f"{model}: {changed_by_model[model]} of {total_by_model[model]} trials changed")
    print(f"re-scored copies are under {out_dir}")


if __name__ == "__main__":
    main()


Overwriting /workspace/qwen35-bench/runpod/rescore_color.py


Validate every config against the datasets (no model is loaded):

In [24]:
bench("runpod/progress.py", "configs/ported_tasks_test_0.8b.yaml", PREFLIGHT_9B[0], "configs/runpod/preflight_27b.yaml",
      "configs/runpod/preflight_27b_upto250.yaml", CONFIG_9B, CONFIG_27B_UPTO250, CONFIG_27B_LARGEST,
      "configs/assoc_inf_ab.yaml")

configs/ported_tasks_test_0.8b.yaml                          18/18   done   failed=0 skipped=0 todo=0
configs/runpod/preflight_9b.yaml                              6/6    done   failed=0 skipped=0 todo=0
configs/runpod/preflight_27b.yaml                             3/6    done   failed=0 skipped=0 todo=3
configs/runpod/preflight_27b_upto250.yaml                     6/6    done   failed=0 skipped=0 todo=0
configs/runpod/qwen35_9b_mst_color.yaml                      35/36   done   failed=0 skipped=0 todo=1
configs/runpod/qwen35_27b_mst_color_upto250.yaml              0/30   done   failed=0 skipped=0 todo=30
configs/runpod/qwen35_27b_mst_color_largest.yaml              0/6    done   failed=0 skipped=0 todo=6
configs/assoc_inf_ab.yaml                                   108/108  done   failed=0 skipped=0 todo=0


## 8. Smoke test (0.8B, a few minutes)

There exists an end-to-end check of the three re-ported tasks, `configs/ported_tasks_test_0.8b.yaml`: 18 cells × 10 trials. It checks that the pipeline runs and scores on this pod, not memory: the 0.8B is a floor model and 10 trials cannot separate anything from chance.

In [25]:
launch("smoke", [("configs/ported_tasks_test_0.8b.yaml", None, GPUS)])
wait("smoke", every=20)

Queue 'smoke': finished   started 2026-10-11T20:58:33, finished 2026-10-11T20:58:42
  [done    ] configs/ported_tasks_test_0.8b.yaml   GPU 0   0:00:09

configs/ported_tasks_test_0.8b.yaml                          18/18   done   failed=0 skipped=0 todo=0
GPU 0: 0% busy, 0 of 140 GiB in use


**Pass if** all 18 cells exist with no ERROR flag, and the replies below are digits (MST 1–3, named color 1–6, associative inference 1–2) or plain angles for continuous color. Accuracy here means nothing. If this cell raises, nothing after it will work: read the job log named in the status above.

In [26]:
bench("runpod/check_results.py", "results/ported_tasks_test", check=False)
show_replies("results/ported_tasks_test", k=2)
n_cells = len(list((REPO / "results/ported_tasks_test").glob("*/*.json")))
problems = result_problems("results/ported_tasks_test")
if n_cells < 18 or problems:
    raise RuntimeError(f"Smoke test incomplete: {n_cells} of 18 cells, problems: {problems[:5]}")
print(f"\nSmoke test passed: {n_cells} of 18 cells, no errors.")


### associative_inference
model         layout    pair  probe      chains   acc       95% CI chance 4B ref wellf agree  flags
qwen3.5-0.8b  original  word  direct_ab       2  0.60 [0.31,0.83]   0.50      -  1.00  0.90  BIAS
qwen3.5-0.8b  original  word  direct_bc       2  0.70 [0.40,0.89]   0.50      -  1.00  0.50  DISAGREE
qwen3.5-0.8b  original  word  inference       2  0.70 [0.40,0.89]   0.50      -  1.00  0.20  BIAS,DISAGREE
qwen3.5-0.8b  original  image direct_ab       2  0.60 [0.31,0.83]   0.50      -  1.00  0.70  BIAS,DISAGREE
qwen3.5-0.8b  original  image direct_bc       2  0.60 [0.31,0.83]   0.50      -  1.00  0.90  BIAS
qwen3.5-0.8b  original  image inference       2  0.50 [0.24,0.76]   0.50      -  1.00  0.70  DISAGREE
qwen3.5-0.8b  original  word  direct_ab       5  0.40 [0.17,0.69]   0.50      -  1.00  0.70  DISAGREE
qwen3.5-0.8b  original  word  direct_bc       5  0.60 [0.31,0.83]   0.50      -  0.90  0.80  
qwen3.5-0.8b  original  word  inference       5  0.50 [0.24,0.7

## 9. Preflight: one trial at the largest prompts

Loads each big model on this pod's hardware and runs one trial per cell at n=1 and at the largest n it will run here: the dataset maximum, or 250 for the 27B on an 80 GB card. It reports peak GPU memory, timing, a time and cost estimate for the full run, and raw replies to eyeball. The 27B once produced gibberish on one cluster setup, so its replies matter as much as its memory.

In [27]:
jobs = []
if RUN_9B:
    jobs.append((PREFLIGHT_9B[0], None, GPUS))
if RUN_27B:
    jobs.append((plan_27b()[1][0], None, GPUS_27B))
if jobs:
    launch("preflight", jobs)
    wait("preflight", every=30)
else:
    print("Neither the 9B nor the 27B is selected; nothing to preflight.")

Queue 'preflight': finished   started 2026-10-11T20:58:59, finished 2026-10-11T21:02:48
  [done    ] configs/runpod/preflight_9b.yaml   GPU 0   0:00:08
  [done    ] configs/runpod/preflight_27b.yaml   GPU 0   0:03:41

configs/runpod/preflight_9b.yaml                              6/6    done   failed=0 skipped=0 todo=0
configs/runpod/preflight_27b.yaml                             6/6    done   failed=0 skipped=0 todo=0
GPU 0: 0% busy, 0 of 140 GiB in use


In [28]:
full = ([CONFIG_9B] if RUN_9B else []) + (plan_27b()[0] if RUN_27B else [])
caps = json.dumps({"qwen3.5-9b": round(capacity(GPUS), 1), "qwen3.5-27b": round(capacity(GPUS_27B), 1)})
bench("runpod/preflight_report.py", "results/runpod_preflight", "--configs", *full,
      "--capacity", caps, "--price", PRICE_PER_HOUR, check=False)
print("\n(The associative-inference gate is not in this estimate: it uses small prompts and takes roughly an hour.)")
show_replies("results/runpod_preflight", k=1)

model         task                         n status    peak GB  cap GB       s   mass  reply
qwen3.5-27b   color_memory/continuous      1 ok           55.0   150.8     0.9      -  '330'
qwen3.5-27b   color_memory/continuous    250 ok           66.7   150.8    42.3      -  '0'
qwen3.5-27b   color_memory/continuous    540 ok           80.3   150.8    42.3      -  '0'
qwen3.5-27b   color_memory/named           1 ok           55.0   150.8     1.4  0.995  '2'
qwen3.5-27b   color_memory/named         250 ok           66.7   150.8    50.2  0.998  '6'
qwen3.5-27b   color_memory/named         540 ok           80.3   150.8    58.2  0.996  '4'
qwen3.5-27b   mst                          1 ok           55.1   150.8    37.7  0.997  '3'
qwen3.5-27b   mst                        250 ok           66.7   150.8    77.9  1.000  '1'
qwen3.5-27b   mst                        576 ok           79.7   150.8    92.0  1.000  '2'
qwen3.5-9b    color_memory/continuous      1 ok           19.0   150.8     0.5      - 

**How to read it.** `FAILED` with `OutOfMemoryError` means the model does not fit at those prompts. For the 27B, set `PART_27B = "up_to_250"` to stop at 250 images on this pod, or use an H200, or set `GPUS_27B = "0,1"` on two 80 GB cards, then re-run the two cells above (finished cells are kept). Replies that are not answers mean: do not run the full job; try the 27B on a single GPU. The next cell holds back any model whose preflight did not pass.

## 10. Full runs (background)

Launches, in order: the associative-inference gate (2B and 4B), the 9B MST + color run, then the 27B: sizes 1 to 250, then MST 576 and color 540 if this pod has room for them (the GPU check says which). You can close the browser; come back and run `status()`. To stop, use `stop("main")` below; finished cells are kept and launching again resumes.

In [29]:
jobs, held = [], []
if RUN_ASSOC_GATE:
    jobs.append(("configs/assoc_inf_ab.yaml", None, GPUS))
configs_27b, (_, sizes_27b), _ = plan_27b()
for on, model, configs, gpus, sizes in ((RUN_9B, "qwen3.5-9b", [CONFIG_9B], GPUS, PREFLIGHT_9B[1]),
                                        (RUN_27B, "qwen3.5-27b", configs_27b, GPUS_27B, sizes_27b)):
    if not on:
        continue
    passed, reasons = preflight_verdict(model, gpus, sizes)
    if passed or FORCE_MAIN:
        jobs += [(config, None, gpus) for config in configs]
    else:
        held.append((model, reasons))
for model, reasons in held:
    print(f"Holding back {model}: its preflight did not pass.")
    for r in reasons:
        print("   -", r)
if jobs:
    launch("main", jobs)
else:
    print("Nothing to launch.")

Started queue 'main' (pid 5267). Jobs, in order:
  configs/runpod/qwen35_9b_mst_color.yaml   GPU 0
  configs/runpod/qwen35_27b_mst_color_upto250.yaml   GPU 0
  configs/runpod/qwen35_27b_mst_color_largest.yaml   GPU 0
Logs: /workspace/runpod_logs


In [30]:
status("main")

Queue 'main': starting...


In [31]:
wait("main", every=300)     # blocks until everything finishes; interrupt any time, the jobs keep running

Queue 'main': finished   started 2026-10-11T21:03:22, finished 2026-10-12T01:05:41
  [done    ] configs/runpod/qwen35_9b_mst_color.yaml   GPU 0   0:09:51
  [done    ] configs/runpod/qwen35_27b_mst_color_upto250.yaml   GPU 0   1:29:10
  [done    ] configs/runpod/qwen35_27b_mst_color_largest.yaml   GPU 0   2:23:18

configs/runpod/qwen35_9b_mst_color.yaml                      36/36   done   failed=0 skipped=0 todo=0
configs/runpod/qwen35_27b_mst_color_upto250.yaml             30/30   done   failed=0 skipped=0 todo=0
configs/runpod/qwen35_27b_mst_color_largest.yaml              6/6    done   failed=0 skipped=0 todo=0
GPU 0: 0% busy, 0 of 140 GiB in use


In [32]:
# Uncomment to stop the full runs (finished cells are kept; launching again resumes):
# stop("main")

## 11. Are the results real?

One row per cell, pooled over seeds, with Wilson 95% intervals and these flags:

| Flag | Meaning | What to do |
|---|---|---|
| `ERROR`, `SKIPPED` | the cell failed or was skipped | read the job log named by `status()`; launching the queue again retries it |
| `GIBBERISH?` | free-text replies are mostly not valid answers and do not match the forced choice | read them below; for the 27B, rerun on a single GPU |
| `DISAGREE` | free text and the forced choice often differ | treat the cell as unreliable |
| `LOW_MASS` | on over 10% of trials the answer digits held under 0.9 of the next-token probability | the choice was read off the tail; unreliable |
| `PARSE` | over 20% of continuous-color replies had no usable angle | read the replies |
| `BIAS` | one answer on over 80% of trials | expected at chance; a problem above it |
| `BELOW_4B?` | over 10 points under the 4B at the same n | suspect the run before the model |

The report and a CSV of every cell are written to `/workspace/outputs`.

In [33]:
dirs = [d for d in ("results/qwen35_9b", "results/qwen35_27b") if (REPO / d).exists()]
bench("runpod/check_results.py", *dirs, "--report", OUT / "RUNPOD_REPORT.md",
      "--csv", OUT / "runpod_checks.csv", check=False)


### color_memory/continuous
model             n  MAE deg  4B ref parse fail within30  flags
qwen3.5-27b       1     13.7      45       0.00     0.90  
qwen3.5-27b       5     21.1       -       0.00     0.82  
qwen3.5-27b      25     19.6       -       0.00     0.81  
qwen3.5-27b     100     32.6       -       0.00     0.61  
qwen3.5-27b     250     50.5       -       0.00     0.45  
qwen3.5-27b     540     65.3       -       0.00     0.36  
qwen3.5-9b        1     24.1      45       0.00     0.80  
qwen3.5-9b        5     34.1       -       0.00     0.62  
qwen3.5-9b       25     49.8       -       0.00     0.50  
qwen3.5-9b      100     55.6       -       0.00     0.40  
qwen3.5-9b      250     62.2       -       0.00     0.34  
qwen3.5-9b      540     68.2       -       0.00     0.28  

### color_memory/named
model             n   acc       95% CI chance 4B ref wellf agree  flags
qwen3.5-27b       1  0.90 [0.83,0.94]   0.17   0.86  1.00  1.00  
qwen3.5-27b       5  0.96 [0.90,0.98]

In [34]:
for d in ("results/qwen35_9b", "results/qwen35_27b"):
    if (REPO / d).exists():
        print(f"===== {d}")
        show_replies(d, k=3)

===== results/qwen35_9b

qwen3.5-9b color_memory continuous degrees   n=540
   target=30.440050134815717 answer=20.0      mass=-      reply='20'
   target=53.34831622652366 answer=180.0     mass=-      reply='180'
   target=204.44671832764186 answer=270.0     mass=-      reply='270'

qwen3.5-9b color_memory named numbered   n=540
   target=orange    answer=purple    mass=0.985  reply='The item is a water bottle in a black protective sleeve. Based on the provided 
   target=purple    answer=purple    mass=0.995  reply='Based on the study sequence, the item was purple.'
   target=red       answer=blue      mass=0.993  reply='The monitor in the image is silver and black. Based on the provided color optio

qwen3.5-9b mst numbered   n=576
   target=old       answer=new       mass=0.997  reply='3'
   target=old       answer=old       mass=0.998  reply='2'
   target=new       answer=new       mass=0.998  reply='2'
===== results/qwen35_27b

qwen3.5-27b color_memory continuous degrees   n=540
 

## 12. Associative-inference gate

Rules in `configs/assoc_inf_ab.yaml` before anyone ran it: a (layout, pair type) is usable only if both direct probes (A→B and B→C, pairs that were actually studied) are above chance with no response bias, for both the 2B and the 4B, at 2 and 5 chains. Only then is the inference probe (A→C, never studied) read as memory. The first port failed this everywhere: the models could not even retrieve a studied pair. If nothing passes again, that is the result, and it belongs in the pickup note rather than a curve.

In [35]:
if (REPO / "results/assoc_inf_ab").exists():
    bench("runpod/assoc_gate.py", "results/assoc_inf_ab", "--json", OUT / "assoc_gate.json",
          "--write-config", "configs/runpod/assoc_inf_full.yaml", check=False)
else:
    print("No gate results yet (RUN_ASSOC_GATE off, or the main queue has not reached it).")

Gate: direct_ab and direct_bc above chance (Wilson lower bound > 0.5) with no BIAS flag,
for both the 2B and 4B, at 2 and 5 chains.

cue_turn  image FAIL
      qwen3.5-2b direct_ab 2ch: 0.58 (low 0.44, top answer 0.88) BIAS
      qwen3.5-2b direct_bc 2ch: 0.58 (low 0.44, top answer 0.88) BIAS
      qwen3.5-2b direct_ab 5ch: 0.52 (low 0.39, top answer 0.88) BIAS
      qwen3.5-2b direct_bc 5ch: 0.52 (low 0.39, top answer 0.96) BIAS
      qwen3.5-4b direct_ab 2ch: 0.58 (low 0.44, top answer 0.84) BIAS
      qwen3.5-4b direct_bc 2ch: 0.54 (low 0.40, top answer 0.56) at chance
      qwen3.5-4b direct_ab 5ch: 0.52 (low 0.39, top answer 0.64) at chance
      qwen3.5-4b direct_bc 5ch: 0.62 (low 0.48, top answer 0.66) at chance
cue_turn  word  FAIL
      qwen3.5-2b direct_ab 2ch: 0.54 (low 0.40, top answer 1.0) BIAS
      qwen3.5-2b direct_bc 2ch: 0.52 (low 0.39, top answer 0.86) BIAS
      qwen3.5-2b direct_ab 5ch: 0.50 (low 0.37, top answer 0.94) BIAS
      qwen3.5-4b direct_ab 2ch: 0.38 (low

In [ ]:
FULL = REPO / "configs/runpod/assoc_inf_full.yaml"
if FULL.exists():
    print(FULL.read_text())
    bench("runpod/progress.py", "configs/runpod/assoc_inf_full.yaml")
    # launch("assoc_full", [("configs/runpod/assoc_inf_full.yaml", None, GPUS)])
else:
    print("No layout passed the gate, so there is no full sweep to run.")

No layout passed the gate, so there is no full sweep to run.


## 13. Figures

Same conventions as`bench.plot`, with her published 4B values as grey crosses for reference.

In [37]:
dirs = [d for d in ("results/qwen35_9b", "results/qwen35_27b", "results/assoc_inf_ab", "results/assoc_inf_full")
        if (REPO / d).exists()]
if dirs:
    bench("runpod/plot_extra.py", *dirs, "--out", "plots/runpod", check=False)
    for png in sorted((REPO / "plots/runpod").glob("*.png")):
        display(Markdown(f"**{png.name}**"))
        display(Image(filename=str(png)))
else:
    print("No results to plot yet.")

Traceback (most recent call last):
  File "/workspace/qwen35-bench/runpod/plot_extra.py", line 18, in <module>
    import matplotlib
  File "/workspace/envs/qwen35/lib/python3.12/site-packages/matplotlib/__init__.py", line 1292, in <module>
    rcParams['backend'] = os.environ.get('MPLBACKEND')
    ~~~~~~~~^^^^^^^^^^^
  File "/workspace/envs/qwen35/lib/python3.12/site-packages/matplotlib/__init__.py", line 784, in __setitem__
    raise ValueError(f"Key {key}: {ve}") from None
ValueError: Key backend: 'module://matplotlib_inline.backend_inline' is not a valid value for backend; supported values are ['gtk3agg', 'gtk3cairo', 'gtk4agg', 'gtk4cairo', 'macosx', 'nbagg', 'notebook', 'qtagg', 'qtcairo', 'qt5agg', 'qt5cairo', 'tkagg', 'tkcairo', 'webagg', 'wx', 'wxagg', 'wxcairo', 'agg', 'cairo', 'pdf', 'pgf', 'ps', 'svg', 'template']


## 14. Final Package

Writes one zip to `/workspace/outputs`. Download it (right-click it in the file browser, then Download), copy it to the cluster, and unzip it **at the repo root**:

- `results/qwen35_9b/{mst,color_memory}/` and `results/qwen35_27b/...` land beside her existing cells. Same run names, so `python -m bench.summarize results/qwen35_9b results/qwen35_27b` pools them.
- `results/assoc_inf_ab/` holds the gate; `configs/runpod/`, `runpod/` and `plots/runpod/` hold the configs, scripts and figures.
- Logs, the check report, and the smoke and preflight results go to `runpod_extras/`, so nothing of hers is overwritten.

Every result file records its GPU, torch and transformers versions. If section 3 restored an earlier pod's zip, this zip holds those results too, so it replaces the earlier one.

In [38]:
stamp = datetime.now().strftime("%Y%m%d_%H%M")
bundle = OUT / f"qwen35_runpod_results_{stamp}.zip"
freeze = OUT / "pip_freeze.txt"      # the environment's exact package versions, for provenance
LIST_PACKAGES = ("import importlib.metadata as m; print(chr(10).join(sorted("
                 "str(d.metadata['Name']) + '==' + d.version for d in m.distributions())))")
r = subprocess.run([str(PY), "-c", LIST_PACKAGES], capture_output=True, text=True)
if r.returncode == 0:
    freeze.write_text(r.stdout)
else:
    print("Could not record the environment's package list; continuing.")

into_repo = ["results/qwen35_9b", "results/qwen35_27b", "results/assoc_inf_ab", "results/assoc_inf_full",
             "configs/runpod", "runpod", "plots/runpod"]
extras = {"results/ported_tasks_test": "runpod_extras/results/ported_tasks_test",
          "results/runpod_preflight": "runpod_extras/results/runpod_preflight"}
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for rel in into_repo:
        for f in sorted((REPO / rel).rglob("*")):
            if f.is_file() and "__pycache__" not in f.parts:
                z.write(f, f.relative_to(REPO).as_posix())
    for rel, dest in extras.items():
        for f in sorted((REPO / rel).rglob("*")):
            if f.is_file():
                z.write(f, f"{dest}/{f.relative_to(REPO / rel).as_posix()}")
    for f in (OUT / "RUNPOD_REPORT.md", OUT / "runpod_checks.csv", OUT / "assoc_gate.json", freeze,
              REPO / "RUNPOD_SOURCE.json"):
        if f.exists():
            z.write(f, f"runpod_extras/{f.name}")
    for f in sorted(LOGS.glob("*")):
        if f.is_file():
            z.write(f, f"runpod_extras/logs/{f.name}")

with zipfile.ZipFile(bundle) as z:
    names = z.namelist()
groups = {}
for n in names:
    parts = n.split("/")
    key = "/".join(parts[:3] if parts[0] == "results" else parts[:2] if parts[0] == "runpod_extras" else parts[:1])
    groups[key] = groups.get(key, 0) + 1
print(f"{bundle}  ({bundle.stat().st_size / 1e6:.1f} MB, {len(names)} files)")
for key, count in sorted(groups.items()):
    print(f"  {key:45s} {count:5d} files")

/workspace/outputs/qwen35_runpod_results_20261012_0106.zip  (1.4 MB, 249 files)
  configs                                           6 files
  results/assoc_inf_ab/associative_inference      108 files
  results/qwen35_27b/color_memory                  24 files
  results/qwen35_27b/mst                           12 files
  results/qwen35_9b/color_memory                   24 files
  results/qwen35_9b/mst                            12 files
  runpod                                            7 files
  runpod_extras/RUNPOD_REPORT.md                    1 files
  runpod_extras/RUNPOD_SOURCE.json                  1 files
  runpod_extras/assoc_gate.json                     1 files
  runpod_extras/logs                               18 files
  runpod_extras/pip_freeze.txt                      1 files
  runpod_extras/results                            33 files
  runpod_extras/runpod_checks.csv                   1 files


## 15. Finishing the 27B on a bigger GPU

Only needed when this pod left out the 27B's two largest sizes (MST 576 and color 540, ~95–100 GB of GPU memory). The cell below checks. If they are still to do:

1. Start a pod with 110 GB+ of GPU memory: one H200, or two 80 GB cards (then set `GPUS_27B = "0,1"`). A 100 GB volume is enough for this one.
2. Upload this notebook, `qwen35-bench-master.zip`, and the results zip from section 14 to its `/workspace`.
3. Set `RUN_9B = False` and `RUN_ASSOC_GATE = False` (they are done), leave `PART_27B = "auto"`, and Run All.

Section 3 restores the finished cells from the zip, so the new pod runs only what is missing (after its own smoke test and preflight). Its zip then holds every result and replaces this one.

In [39]:
if not RUN_27B:
    print("RUN_27B is off on this pod, so none of the 27B ran here. The steps above run all of it on a bigger GPU.")
else:
    r = subprocess.run([str(PY), "runpod/progress.py", "--json", CONFIG_27B_UPTO250, CONFIG_27B_LARGEST],
                       cwd=REPO, env=ENV, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout + r.stderr)
    else:
        upto250, largest = json.loads(r.stdout)
        print(f"27B, sizes 1 to 250:          {upto250['done']} of {upto250['total']} result files done")
        print(f"27B, MST 576 and color 540:   {largest['done']} of {largest['total']} result files done")
        if upto250["done"] == upto250["total"] and largest["done"] == largest["total"]:
            print("\nThe 27B is complete. Nothing to finish.")
        elif upto250["done"] < upto250["total"]:
            print("\nThe sizes up to 250 are not all done either: see status('main'), or the preflight if the 27B "
                  "was held back. The steps above also run whatever is missing.")
        elif not plan_27b()[2]:
            print("\nOnly MST 576 and color 540 are left, and this pod is too small for them. "
                  "Finish them on a bigger GPU with the steps above.")
        else:
            print("\nThis pod is set to run them: check status('main') for failures, then launch again (section 10).")

27B, sizes 1 to 250:          30 of 30 result files done
27B, MST 576 and color 540:   6 of 6 result files done

The 27B is complete. Nothing to finish.


## Appendix A. Re-score the old continuous-color results (optional, CPU only)

Open item 59 in: the 0.8B, 2B and 4B continuous-color cells were scored with a parser that took the *first* number in a reply; the corrected parser takes the last number in [0, 360], as vision-memory does. Re-scoring changes 188 of 1,600 trials (0.8B 105, 2B 83, 4B 0). To do it here, upload her `results/qwen35_{0.8b,2b,4b}/color_memory/` folders into `/workspace/rescore_in/` (keeping the folder names) and run the cell. Corrected copies go to `/workspace/rescore_out/`; the originals are untouched.

In [40]:
RESCORE_IN, RESCORE_OUT = WORK / "rescore_in", WORK / "rescore_out"
if RESCORE_IN.exists():
    bench("runpod/rescore_color.py", RESCORE_IN, RESCORE_OUT)
else:
    print(f"Nothing to re-score: upload the old color_memory result folders into {RESCORE_IN} first.")

Nothing to re-score: upload the old color_memory result folders into /workspace/rescore_in first.


## Appendix B. Stop the pod

Download the zip first. **Stop** keeps the `/workspace` volume (it still bills for storage) and frees the GPU; **Terminate** deletes everything not on a network volume. You can do either from the RunPod console, or uncomment the line below.

In [41]:
pod = os.environ.get("RUNPOD_POD_ID")
print("Pod id:", pod or "unknown (not on RunPod?)")
# run(["runpodctl", "stop", "pod", pod])

Pod id: cslyvbh4d8v8ac


## Troubleshooting

- **"CUDA driver version is insufficient" or "no kernel image is available"**: the pod's driver is older than 570. Redeploy with CUDA 12.8 or newer.
- **`OutOfMemoryError`** in the preflight, or a `FAILED` job in `status()`: the model does not fit that GPU at those prompts. For the 27B, set `PART_27B = "up_to_250"`, or use an H200 or `GPUS_27B = "0,1"`, then launch again; finished cells are kept.
- **Model download fails with 401 or 403**: set `HF_TOKEN` in the pod's environment variables (or the settings cell) and re-run the weights cell.
- **The first trial is very slow**: the linear-attention kernels compile on first use (20–140 s). They are cached in `/workspace/.triton` afterwards.
- **The pod restarted**: run the settings and helpers cells, then `status("main")`. If the queue is gone, run the launch cell again; it resumes.
- **The pod is gone** (terminated, or you moved to another GPU): start a new pod, upload the notebook, the repo zip and your latest results zip, and Run All. Section 3 restores the finished cells, so only what is missing runs.
- **27B replies are gibberish**: on the cluster this happened once with the 27B sharded over eight GPUs. Run it on one H200.
- **You want a job's full log**: open the file named by `status()` in `/workspace/runpod_logs`.